# FloodShift 10 — Final Reliable Grounded Variational Vision-Language Planner

This notebook is the **final VLM redesign**. It deliberately avoids an endless 09A/09B/09C loop.

It keeps the parts that already worked and replaces the parts that did not.

## Scientific motivation

The design follows five well-established findings from published VLM work:

1. **High-resolution patch features matter for remote sensing.** GeoChat adapts the LLaVA architecture to remote sensing using CLIP ViT-L/14 and high-resolution visual tokens rather than one pooled feature.
2. **Instruction-aware visual extraction is stronger than instruction-agnostic pooling.** InstructBLIP conditions its visual query module on the user instruction.
3. **Lightweight vision-language connectors can be very effective.** LLaVA-1.5 reports strong data-efficient results using a two-layer MLP connector with CLIP-L/336 features.
4. **VLM uncertainty should not be trusted solely from verbalized self-confidence.** We retain FloodShift's explicitly supervised Gaussian uncertainty branch.
5. **Exact scientific facts should be structurally constrained.** The local multimodal model predicts a finite response plan; a deterministic renderer inserts verified GeoEngine facts. The later live LLM may phrase those verified facts, but it is not allowed to invent them.

## What is reused

### Accepted scientific detector stack
- FloodShift-04H-CAL
- calibrated predictive uncertainty
- counterfactual sensor attribution
- deterministic GeoEngine
- accepted FloodShift-08B corpus

### Successful 09B component
The 09B Gaussian scene encoder achieved strong held-out entropy alignment and is reused **as a frozen visual uncertainty branch**.

Expected 09B archive SHA256:

```text
30c1de0205f76eb5e41223550e8c093525eaaca024794f8b6a3f30aeee6267ce
```

## Final architecture

```text
Evidence panel
   │
   ├── CLIP ViT-L/14 336 frozen high-resolution patch features
   │       ↓
   │   instruction-aware query resampler
   │
   └── 09B frozen Gaussian scene encoder
           ↓
       μ, logσ², semantic predictions

             ↓
     learned visual semantic tokens
             ↓
        FLAN-T5 encoder + LoRA
             ↓
       finite response-plan class
             ↓
    deterministic verified renderer
             ↓
       grounded final response
```

This is intentionally a **vision-language planner**, not an unconstrained numerical text generator.

The scientific numbers stay owned by the deterministic GeoEngine.

## Visual grounding protocol

The core visual-only tasks are:

- temporary-flood extent descriptor
- uncertainty descriptor

Their class boundaries are computed **only from the 314 training scenes using tertiles**. They are descriptive bins, not emergency-risk thresholds.

Review and sensor-dominance tasks are evaluated separately because their natural class distributions are strongly imbalanced.

We use:
- full visual input,
- zero visual input,
- true scene-level shuffled visual input.

## Final test rule

The Pakistan + Spain geographic test remains:

```text
LOCKED_NOT_EVALUATED
```

This notebook never reads those 58 scenes.


In [13]:
# ============================================================
# P20-C00 — KAGGLE DEPENDENCIES
# Internet ON.
# ============================================================
!pip uninstall -y torchao >/dev/null 2>&1
!pip -q install -U "transformers>=4.46,<5" "peft>=0.13,<1" accelerate sentencepiece safetensors

print("P20-C00 COMPLETE ✅")


P20-C00 COMPLETE ✅


In [14]:
# ============================================================
# P20-C01 — ENVIRONMENT
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, math, time, random, re, hashlib, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    CLIPImageProcessor,
    CLIPVisionModel,
    T5EncoderModel,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

# 09A showed T4 FP16 instability with this family.
# Keep the final training protocol numerically conservative.
AMP_ENABLED = False

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_10_Final_VLP")
SOURCE08B = OUTPUT_ROOT / "_source08b"
SOURCE09B = OUTPUT_ROOT / "_source09b"
CACHE_ROOT = OUTPUT_ROOT / "cache"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoint"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [
    OUTPUT_ROOT,
    SOURCE08B,
    SOURCE09B,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    METRIC_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_08B_SHA = (
    "421bb74db41f90012b7e5d0f1b342ecfc467b657b12177806973f3bf2e13da20"
)

EXPECTED_09B_SHA = (
    "30c1de0205f76eb5e41223550e8c093525eaaca024794f8b6a3f30aeee6267ce"
)

BASE_VISION_NAME = "openai/clip-vit-base-patch32"
HIGHRES_VISION_NAME = "openai/clip-vit-large-patch14-336"
TEXT_ENCODER_NAME = "google/flan-t5-base"

print("=" * 112)
print("P20-C01 — ENVIRONMENT")
print("=" * 112)
print("Python         :", sys.version.split()[0])
print("PyTorch        :", torch.__version__)
print("Device         :", DEVICE)
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
print("AMP            :", AMP_ENABLED)
print("High-res vision:", HIGHRES_VISION_NAME)
print("Text encoder   :", TEXT_ENCODER_NAME)
print("Geo test       : LOCKED_NOT_EVALUATED")
print("P20-C01 COMPLETE ✅")


P20-C01 — ENVIRONMENT
Python         : 3.12.13
PyTorch        : 2.10.0+cu128
Device         : cuda:0
GPU            : Tesla T4
AMP            : False
High-res vision: openai/clip-vit-large-patch14-336
Text encoder   : google/flan-t5-base
Geo test       : LOCKED_NOT_EVALUATED
P20-C01 COMPLETE ✅


In [15]:
# ============================================================
# P20-C02 — LOCATE + VERIFY EXPANDED 08B + 09B DATASETS
# Kaggle has already unpacked the uploaded archives.
# ============================================================

from pathlib import Path
import json
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")


# ------------------------------------------------------------
# Helper: find an exact file anywhere under /kaggle/input
# ------------------------------------------------------------

def find_exact(filename):
    matches = sorted(
        INPUT_ROOT.rglob(filename),
        key=lambda p: (len(str(p)), str(p)),
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not find:\n{filename}\n"
            f"under /kaggle/input"
        )

    if len(matches) > 1:
        print(f"⚠️ Multiple matches for {filename}")

        for p in matches:
            print("  ", p)

        print("Using:", matches[0])

    return matches[0]


# ============================================================
# 08B — DIRECTLY USE EXPANDED DATASET CONTENTS
# ============================================================

TRAIN_JSONL = find_exact(
    "FloodShift_08B_train.jsonl"
)

VAL_JSONL = find_exact(
    "FloodShift_08B_validation.jsonl"
)

TRAIN_PACKETS_JSONL = find_exact(
    "FloodShift_08B_train_scene_packets.jsonl"
)

VAL_PACKETS_JSONL = find_exact(
    "FloodShift_08B_validation_scene_packets.jsonl"
)

DATASET_CARD_PATH = find_exact(
    "FloodShift_08B_dataset_card.json"
)

PROTOCOL_08B_PATH = find_exact(
    "FloodShift_08B_protocol_record.json"
)

MANIFEST_08B_PATH = find_exact(
    "FloodShift_08B_file_manifest.csv"
)


# IMPORTANT:
# P20-C03 later searches SOURCE08B for *_evidence.png.
# Therefore point SOURCE08B directly to the mounted 08B dataset.
SOURCE08B = DATASET_CARD_PATH.parent


# ------------------------------------------------------------
# Verify 08B metadata
# ------------------------------------------------------------

with open(
    DATASET_CARD_PATH,
    "r",
    encoding="utf-8",
) as f:
    dataset_card = json.load(f)


with open(
    PROTOCOL_08B_PATH,
    "r",
    encoding="utf-8",
) as f:
    protocol08b = json.load(f)


assert dataset_card["training_scenes"] == 314
assert dataset_card["validation_scenes"] == 34
assert dataset_card["training_examples"] == 3140
assert dataset_card["validation_examples"] == 340

assert (
    dataset_card[
        "scene_overlap_train_validation"
    ]
    == 0
)

assert (
    dataset_card[
        "test_status"
    ]
    == "LOCKED_NOT_EVALUATED"
)

assert (
    protocol08b[
        "test_status"
    ]
    == "LOCKED_NOT_EVALUATED"
)


# ------------------------------------------------------------
# Confirm evidence panels are present
# ------------------------------------------------------------

evidence_panels = list(
    SOURCE08B.rglob(
        "*_evidence.png"
    )
)

assert len(evidence_panels) == 348, (
    f"Expected 348 evidence panels, "
    f"found {len(evidence_panels)}"
)


# ============================================================
# 09B — DIRECTLY USE EXPANDED DATASET CONTENTS
# ============================================================

GAUSSIAN_09B_PATH = find_exact(
    "FloodShift_09B_GaussianSceneEncoder.pt"
)

PROTOCOL_09B_PATH = find_exact(
    "FloodShift_09B_protocol_record.json"
)

MANIFEST_09B_PATH = find_exact(
    "FloodShift_09B_file_manifest.csv"
)


# Point SOURCE09B at actual Kaggle-mounted folder.
SOURCE09B = PROTOCOL_09B_PATH.parent


with open(
    PROTOCOL_09B_PATH,
    "r",
    encoding="utf-8",
) as f:
    protocol09b = json.load(f)


assert (
    protocol09b[
        "frozen_geographic_test"
    ]
    == "LOCKED_NOT_EVALUATED"
)


# ============================================================
# BASIC MANIFEST CHECKS
# ============================================================

manifest08b = pd.read_csv(
    MANIFEST_08B_PATH
)

manifest09b = pd.read_csv(
    MANIFEST_09B_PATH
)

assert len(manifest08b) > 0
assert len(manifest09b) > 0


# ============================================================
# FINAL REPORT
# ============================================================

print("=" * 112)
print("P20-C02 — INPUTS VERIFIED")
print("=" * 112)

print()
print("08B")
print("-" * 60)

print(
    "Mounted root       :",
    SOURCE08B,
)

print(
    "Training JSONL     :",
    TRAIN_JSONL,
)

print(
    "Validation JSONL   :",
    VAL_JSONL,
)

print(
    "Train packets      :",
    TRAIN_PACKETS_JSONL,
)

print(
    "Validation packets :",
    VAL_PACKETS_JSONL,
)

print(
    "Evidence panels    :",
    len(evidence_panels),
)

print(
    "Manifest entries   :",
    len(manifest08b),
)


print()
print("09B")
print("-" * 60)

print(
    "Mounted root       :",
    SOURCE09B,
)

print(
    "Gaussian encoder   :",
    GAUSSIAN_09B_PATH,
)

print(
    "Protocol           :",
    PROTOCOL_09B_PATH,
)

print(
    "Manifest entries   :",
    len(manifest09b),
)


print()
print("Scientific protocol")
print("-" * 60)

print(
    "Train scenes       :",
    dataset_card["training_scenes"],
)

print(
    "Validation scenes  :",
    dataset_card["validation_scenes"],
)

print(
    "Scene overlap      :",
    dataset_card["scene_overlap_train_validation"],
)

print(
    "Geo test           : LOCKED_NOT_EVALUATED"
)

print()
print("P20-C02 COMPLETE ✅")

P20-C02 — INPUTS VERIFIED

08B
------------------------------------------------------------
Mounted root       : /kaggle/input/datasets/himavarshini2108/floodshift-08bb-archive-zip/dataset
Training JSONL     : /kaggle/input/datasets/himavarshini2108/floodshift-08bb-archive-zip/dataset/FloodShift_08B_train.jsonl
Validation JSONL   : /kaggle/input/datasets/himavarshini2108/floodshift-08bb-archive-zip/dataset/FloodShift_08B_validation.jsonl
Train packets      : /kaggle/input/datasets/himavarshini2108/floodshift-08bb-archive-zip/dataset/FloodShift_08B_train_scene_packets.jsonl
Validation packets : /kaggle/input/datasets/himavarshini2108/floodshift-08bb-archive-zip/dataset/FloodShift_08B_validation_scene_packets.jsonl
Evidence panels    : 348
Manifest entries   : 359

09B
------------------------------------------------------------
Mounted root       : /kaggle/input/datasets/himavarshini2108/floodshift-09b-archive-zip
Gaussian encoder   : /kaggle/input/datasets/himavarshini2108/floodshift-0

In [16]:
# ============================================================
# P20-C03 — LOAD CORPUS / PACKETS / PANELS
# ============================================================
def read_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows

train_examples = read_jsonl(TRAIN_JSONL)
val_examples = read_jsonl(VAL_JSONL)
train_packets = read_jsonl(TRAIN_PACKETS_JSONL)
val_packets = read_jsonl(VAL_PACKETS_JSONL)

assert len(train_examples) == 3140
assert len(val_examples) == 340
assert len(train_packets) == 314
assert len(val_packets) == 34

packet_by_scene = {
    p["scene_id"]: p
    for p in train_packets + val_packets
}

train_scene_ids = sorted({
    x["scene_id"]
    for x in train_examples
})

val_scene_ids = sorted({
    x["scene_id"]
    for x in val_examples
})

assert len(train_scene_ids) == 314
assert len(val_scene_ids) == 34
assert set(train_scene_ids).isdisjoint(val_scene_ids)

all_panels = list(
    SOURCE08B.rglob("*_evidence.png")
)

panel_by_scene = {}

for p in all_panels:
    suffix = "_evidence.png"

    if not p.name.endswith(suffix):
        continue

    sid = p.name[:-len(suffix)]

    if (
        sid in packet_by_scene
        and sid not in panel_by_scene
    ):
        panel_by_scene[sid] = p

assert len(panel_by_scene) == 348, (
    f"Expected 348 evidence panels, found {len(panel_by_scene)}"
)

def true_derangement(scene_ids):
    ids = list(sorted(scene_ids))

    return {
        ids[i]: ids[(i + 1) % len(ids)]
        for i in range(len(ids))
    }

TRAIN_DERANGEMENT = true_derangement(train_scene_ids)
VAL_DERANGEMENT = true_derangement(val_scene_ids)

for sid in val_scene_ids:
    assert VAL_DERANGEMENT[sid] != sid

print("=" * 112)
print("P20-C03 — CORPUS")
print("=" * 112)
print("Train scenes     :", len(train_scene_ids))
print("Validation scenes:", len(val_scene_ids))
print("Train examples   :", len(train_examples))
print("Validation       :", len(val_examples))
print("Panels           :", len(panel_by_scene))
print("Scene overlap    :", len(set(train_scene_ids) & set(val_scene_ids)))
print("P20-C03 COMPLETE ✅")


P20-C03 — CORPUS
Train scenes     : 314
Validation scenes: 34
Train examples   : 3140
Validation       : 340
Panels           : 348
Scene overlap    : 0
P20-C03 COMPLETE ✅


In [17]:
# ============================================================
# P20-C04 — TRAIN-ONLY DESCRIPTIVE BINS + SCENE TARGETS
# ============================================================
def parse_review_label(scene_id, examples):
    rows = [
        x for x in examples
        if x["scene_id"] == scene_id
        and x["task"] == "review_policy"
    ]

    assert len(rows) == 1

    t = rows[0]["response"].strip().lower()

    if t.startswith("yes"):
        return 1

    if t.startswith("no"):
        return 0

    raise ValueError(
        f"Could not parse review response: {rows[0]['response']}"
    )

train_review = {
    sid: parse_review_label(
        sid,
        train_examples,
    )
    for sid in train_scene_ids
}

val_review = {
    sid: parse_review_label(
        sid,
        val_examples,
    )
    for sid in val_scene_ids
}

def packet_continuous_targets(scene_id):
    p = packet_by_scene[scene_id]

    temp = float(
        p["segmentation"]["temporary_flood"]["fraction_of_coverage"]
    )

    entropy = float(
        p["uncertainty"]["mean_entropy"]
    )

    optical = float(
        p["counterfactual_attribution"]["optical_share"]
    )

    return (
        temp,
        entropy,
        optical,
    )

train_temp = np.asarray([
    packet_continuous_targets(sid)[0]
    for sid in train_scene_ids
])

train_entropy = np.asarray([
    packet_continuous_targets(sid)[1]
    for sid in train_scene_ids
])

# Tertiles are computed from TRAIN only.
EXTENT_BOUNDS = tuple(
    float(x)
    for x in np.quantile(
        train_temp,
        [1/3, 2/3],
    )
)

UNCERTAINTY_BOUNDS = tuple(
    float(x)
    for x in np.quantile(
        train_entropy,
        [1/3, 2/3],
    )
)

def three_bin(x, bounds):
    a, b = bounds

    if x <= a:
        return 0

    if x <= b:
        return 1

    return 2

def normalize_dominance(value):
    t = str(value).lower()

    if "optical" in t:
        return 0

    if "sar" in t:
        return 2

    return 1

def scene_targets(scene_id, split):
    p = packet_by_scene[scene_id]

    temp, entropy, optical = packet_continuous_targets(
        scene_id
    )

    review = (
        train_review[scene_id]
        if split == "train"
        else val_review[scene_id]
    )

    dominance = normalize_dominance(
        p["counterfactual_attribution"]["dominance"]
    )

    return {
        "continuous": np.asarray(
            [
                temp,
                entropy,
                optical,
            ],
            dtype=np.float32,
        ),
        "extent_class": int(
            three_bin(
                temp,
                EXTENT_BOUNDS,
            )
        ),
        "uncertainty_class": int(
            three_bin(
                entropy,
                UNCERTAINTY_BOUNDS,
            )
        ),
        "review_class": int(
            review
        ),
        "dominance_class": int(
            dominance
        ),
    }

print("=" * 112)
print("P20-C04 — TRAIN-ONLY DESCRIPTIVE BINS")
print("=" * 112)
print("Extent tertiles     :", EXTENT_BOUNDS)
print("Uncertainty tertiles:", UNCERTAINTY_BOUNDS)
print()
print("Train extent counts:")
print(
    Counter(
        scene_targets(s, "train")["extent_class"]
        for s in train_scene_ids
    )
)
print("Val extent counts:")
print(
    Counter(
        scene_targets(s, "val")["extent_class"]
        for s in val_scene_ids
    )
)
print("Train uncertainty counts:")
print(
    Counter(
        scene_targets(s, "train")["uncertainty_class"]
        for s in train_scene_ids
    )
)
print("Val uncertainty counts:")
print(
    Counter(
        scene_targets(s, "val")["uncertainty_class"]
        for s in val_scene_ids
    )
)
print("P20-C04 COMPLETE ✅")


P20-C04 — TRAIN-ONLY DESCRIPTIVE BINS
Extent tertiles     : (0.008143621010519039, 0.04433379074221727)
Uncertainty tertiles: (0.04018604134519895, 0.08872490127881368)

Train extent counts:
Counter({2: 105, 0: 105, 1: 104})
Val extent counts:
Counter({2: 17, 0: 9, 1: 8})
Train uncertainty counts:
Counter({2: 105, 0: 105, 1: 104})
Val uncertainty counts:
Counter({2: 17, 1: 10, 0: 7})
P20-C04 COMPLETE ✅


In [18]:
# ============================================================
# P20-C05 — RECREATE FROZEN 09B GAUSSIAN BRANCH
# Uses the original CLIP-B/32 feature definition.
# ============================================================
base_processor = CLIPImageProcessor.from_pretrained(
    BASE_VISION_NAME
)

base_vision = CLIPVisionModel.from_pretrained(
    BASE_VISION_NAME
).to(DEVICE).eval()

for p in base_vision.parameters():
    p.requires_grad = False

@torch.inference_mode()
def base_features(path):
    with Image.open(path) as img:
        img = img.convert("RGB")

        batch = base_processor(
            images=img,
            return_tensors="pt",
        )

    out = base_vision(
        pixel_values=batch["pixel_values"].to(DEVICE),
        return_dict=True,
    )

    pooled = out.pooler_output[0].float()

    patch = out.last_hidden_state[
        0,
        1:,
        :
    ].float()

    n, d = patch.shape
    side = int(round(math.sqrt(n)))

    assert side * side == n

    grid = (
        patch
        .reshape(
            side,
            side,
            d,
        )
        .permute(
            2,
            0,
            1,
        )
        .unsqueeze(0)
    )

    patch9 = (
        F.adaptive_avg_pool2d(
            grid,
            (3, 3),
        )[0]
        .permute(
            1,
            2,
            0,
        )
        .reshape(
            9,
            d,
        )
    )

    return (
        pooled.cpu(),
        patch9.cpu(),
    )

BASE_CACHE = {}

for i, sid in enumerate(
    sorted(panel_by_scene),
    start=1,
):
    pooled, patch9 = base_features(
        panel_by_scene[sid]
    )

    BASE_CACHE[sid] = {
        "pooled": pooled,
        "patch9": patch9,
    }

    if (
        i == 1
        or i % 75 == 0
        or i == len(panel_by_scene)
    ):
        print(
            f"[{i:03d}/{len(panel_by_scene)}] CLIP-B/32"
        )

del base_vision
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASE_DIM = int(
    next(iter(BASE_CACHE.values()))["pooled"].numel()
)

assert BASE_DIM == 768

LATENT_DIM = 256
LOGVAR_MIN = -6.0
LOGVAR_MAX = 1.0

class GaussianSceneEncoder(nn.Module):
    def __init__(
        self,
        vision_dim,
        latent_dim=LATENT_DIM,
    ):
        super().__init__()

        self.scene_proj = nn.Sequential(
            nn.LayerNorm(
                vision_dim * 2
            ),
            nn.Linear(
                vision_dim * 2,
                512,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.mu = nn.Linear(
            512,
            latent_dim,
        )

        self.logvar = nn.Linear(
            512,
            latent_dim,
        )

        self.regression_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                128,
            ),
            nn.GELU(),
            nn.Linear(
                128,
                3,
            ),
        )

        self.dominance_head = nn.Linear(
            latent_dim,
            3,
        )

        self.review_head = nn.Linear(
            latent_dim,
            1,
        )

    def forward(
        self,
        pooled,
        patch9,
        sample=False,
    ):
        patch_mean = patch9.mean(
            dim=1
        )

        scene_input = torch.cat(
            [
                pooled.float(),
                patch_mean.float(),
            ],
            dim=-1,
        )

        h = self.scene_proj(
            scene_input
        )

        mu = self.mu(
            h
        )

        logvar = self.logvar(
            h
        ).clamp(
            LOGVAR_MIN,
            LOGVAR_MAX,
        )

        if sample:
            std = torch.exp(
                0.5 * logvar
            )

            z = (
                mu
                + std
                * torch.randn_like(
                    std
                )
            )
        else:
            z = mu

        regression = torch.sigmoid(
            self.regression_head(
                mu
            )
        )

        dominance_logits = self.dominance_head(
            mu
        )

        review_logit = self.review_head(
            mu
        ).squeeze(-1)

        latent_uncertainty = torch.sigmoid(
            logvar.mean(
                dim=1
            )
        )

        return {
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "regression": regression,
            "dominance_logits": dominance_logits,
            "review_logit": review_logit,
            "latent_uncertainty": latent_uncertainty,
        }

gaussian_encoder = GaussianSceneEncoder(
    BASE_DIM
)

gaussian_ckpt = torch.load(
    GAUSSIAN_09B_PATH,
    map_location="cpu",
    weights_only=False,
)

gaussian_encoder.load_state_dict(
    gaussian_ckpt["state_dict"],
    strict=True,
)

gaussian_encoder = gaussian_encoder.to(
    DEVICE
).eval()

for p in gaussian_encoder.parameters():
    p.requires_grad = False

print("=" * 112)
print("P20-C05 — 09B GAUSSIAN RELOADED")
print("=" * 112)
print("Checkpoint validation:")
print(
    json.dumps(
        gaussian_ckpt.get(
            "validation",
            {},
        ),
        indent=2,
    )
)
print("P20-C05 COMPLETE ✅")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

[001/348] CLIP-B/32
[075/348] CLIP-B/32
[150/348] CLIP-B/32
[225/348] CLIP-B/32
[300/348] CLIP-B/32
[348/348] CLIP-B/32
P20-C05 — 09B GAUSSIAN RELOADED
Checkpoint validation:
{
  "temp_fraction_MAE": 0.04811626300215721,
  "entropy_aux_MAE": 0.018693624064326286,
  "optical_share_MAE": 0.10007955878973007,
  "latent_uncertainty_MAE": 0.019725775346159935,
  "latent_entropy_correlation": 0.8414468513366232,
  "dominance_accuracy": 0.7352941176470589,
  "review_accuracy": 0.7352941176470589
}
P20-C05 COMPLETE ✅


In [19]:
# ============================================================
# P20-C06 — REVALIDATE GAUSSIAN UNCERTAINTY
# Then cache its learned visual semantics for all 348 scenes.
# ============================================================
GAUSSIAN_CACHE = {}

@torch.inference_mode()
def gaussian_scene_output(scene_id):
    v = BASE_CACHE[
        scene_id
    ]

    out = gaussian_encoder(
        v["pooled"].unsqueeze(0).to(DEVICE),
        v["patch9"].unsqueeze(0).to(DEVICE),
        sample=False,
    )

    return {
        k: tensor[0].detach().cpu()
        for k, tensor in out.items()
    }

for sid in sorted(panel_by_scene):
    GAUSSIAN_CACHE[sid] = gaussian_scene_output(
        sid
    )

val_latent_unc = np.asarray([
    float(
        GAUSSIAN_CACHE[sid][
            "latent_uncertainty"
        ]
    )
    for sid in val_scene_ids
])

val_true_entropy = np.asarray([
    scene_targets(sid, "val")[
        "continuous"
    ][1]
    for sid in val_scene_ids
])

gaussian_entropy_r = float(
    np.corrcoef(
        val_latent_unc,
        val_true_entropy,
    )[0, 1]
)

gaussian_entropy_mae = float(
    np.mean(
        np.abs(
            val_latent_unc
            - val_true_entropy
        )
    )
)

assert np.isfinite(
    gaussian_entropy_r
)

print("=" * 112)
print("P20-C06 — GAUSSIAN REVALIDATION")
print("=" * 112)
print("Entropy correlation:", gaussian_entropy_r)
print("Entropy MAE        :", gaussian_entropy_mae)
print("Expected strong 09B behavior retained.")
print("P20-C06 COMPLETE ✅")


P20-C06 — GAUSSIAN REVALIDATION
Entropy correlation: 0.8414468577487988
Entropy MAE        : 0.0197257731548127
Expected strong 09B behavior retained.
P20-C06 COMPLETE ✅


In [20]:
# ============================================================
# P20-C07 — HIGH-RES CLIP-L/14-336 SPATIAL CACHE
#
# Preserve much more spatial information than 09A/09B.
# 24x24 native patch grid -> deterministic 12x12 grid.
# ============================================================
highres_processor = CLIPImageProcessor.from_pretrained(
    HIGHRES_VISION_NAME
)

highres_vision = CLIPVisionModel.from_pretrained(
    HIGHRES_VISION_NAME
).to(DEVICE).eval()

for p in highres_vision.parameters():
    p.requires_grad = False

@torch.inference_mode()
def highres_features(path):
    with Image.open(path) as img:
        img = img.convert("RGB")

        batch = highres_processor(
            images=img,
            return_tensors="pt",
        )

    out = highres_vision(
        pixel_values=batch[
            "pixel_values"
        ].to(DEVICE),
        return_dict=True,
    )

    pooled = out.pooler_output[
        0
    ].float()

    patch = out.last_hidden_state[
        0,
        1:,
        :
    ].float()

    n, d = patch.shape
    side = int(
        round(
            math.sqrt(n)
        )
    )

    assert side * side == n

    grid = (
        patch
        .reshape(
            side,
            side,
            d,
        )
        .permute(
            2,
            0,
            1,
        )
        .unsqueeze(0)
    )

    patch12 = (
        F.adaptive_avg_pool2d(
            grid,
            (12, 12),
        )[0]
        .permute(
            1,
            2,
            0,
        )
        .reshape(
            144,
            d,
        )
    )

    return (
        pooled.cpu(),
        patch12.half().cpu(),
    )

HIGHRES_CACHE = {}

t0 = time.time()

for i, sid in enumerate(
    sorted(panel_by_scene),
    start=1,
):
    pooled, patch12 = highres_features(
        panel_by_scene[sid]
    )

    HIGHRES_CACHE[sid] = {
        "pooled": pooled,
        "patch12": patch12,
    }

    if (
        i == 1
        or i % 50 == 0
        or i == len(panel_by_scene)
    ):
        print(
            f"[{i:03d}/{len(panel_by_scene)}] CLIP-L/336"
        )

HIGHRES_DIM = int(
    next(
        iter(
            HIGHRES_CACHE.values()
        )
    )["pooled"].numel()
)

PATCH_TOKENS = int(
    next(
        iter(
            HIGHRES_CACHE.values()
        )
    )["patch12"].shape[0]
)

del highres_vision
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 112)
print("P20-C07 — HIGH-RES CACHE")
print("=" * 112)
print("Scenes      :", len(HIGHRES_CACHE))
print("Feature dim :", HIGHRES_DIM)
print("Patch tokens:", PATCH_TOKENS)
print("Minutes     :", round((time.time() - t0) / 60.0, 2))
print("P20-C07 COMPLETE ✅")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

[001/348] CLIP-L/336
[050/348] CLIP-L/336
[100/348] CLIP-L/336
[150/348] CLIP-L/336
[200/348] CLIP-L/336
[250/348] CLIP-L/336
[300/348] CLIP-L/336
[348/348] CLIP-L/336
P20-C07 — HIGH-RES CACHE
Scenes      : 348
Feature dim : 1024
Patch tokens: 144
Minutes     : 0.98
P20-C07 COMPLETE ✅


In [21]:
# ============================================================
# P20-C08 — LOAD FROZEN FLAN-T5 ENCODER
#
# Its word embeddings provide instruction context to the visual
# query module, following the instruction-aware principle.
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_ENCODER_NAME
)

text_encoder = T5EncoderModel.from_pretrained(
    TEXT_ENCODER_NAME
)

TEXT_DIM = int(
    text_encoder.config.d_model
)

# Freeze initially while visual alignment is pretrained.
text_encoder = text_encoder.to(
    DEVICE
).eval()

for p in text_encoder.parameters():
    p.requires_grad = False

VISUAL_TASK_PROMPTS = {
    "extent": (
        "Classify the temporary-flood extent visible in this scene."
    ),
    "uncertainty": (
        "Classify the FloodShift uncertainty descriptor for this scene."
    ),
    "review": (
        "Classify whether this scene is flagged by the fixed review policy."
    ),
    "dominance": (
        "Classify the counterfactual sensor-dependence dominance."
    ),
}

@torch.inference_mode()
def instruction_context(texts):
    batch = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=96,
        return_tensors="pt",
    )

    ids = batch[
        "input_ids"
    ].to(DEVICE)

    mask = batch[
        "attention_mask"
    ].to(DEVICE)

    emb = text_encoder.get_input_embeddings()(
        ids
    )

    m = mask.unsqueeze(-1).float()

    mean = (
        (
            emb
            * m
        ).sum(
            dim=1
        )
        / m.sum(
            dim=1
        ).clamp_min(
            1.0
        )
    )

    return mean.detach()

TASK_CONTEXT = {
    name: instruction_context(
        [prompt]
    )[0].cpu()
    for name, prompt in VISUAL_TASK_PROMPTS.items()
}

print("=" * 112)
print("P20-C08 — LANGUAGE CONTEXT")
print("=" * 112)
print("T5 d_model:", TEXT_DIM)
print("Task contexts:", list(TASK_CONTEXT))
print("P20-C08 COMPLETE ✅")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

P20-C08 — LANGUAGE CONTEXT
T5 d_model: 768
Task contexts: ['extent', 'uncertainty', 'review', 'dominance']
P20-C08 COMPLETE ✅


In [22]:
# ============================================================
# P20-C09 — INSTRUCTION-AWARE VISUAL QUERY RESAMPLER
#
# Inspired by instruction-aware visual querying:
# learned queries + instruction conditioning + cross-attention.
# ============================================================
VISUAL_HIDDEN = 512
N_QUERIES = 8
N_HEADS = 8
N_LAYERS = 2

class CrossAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        heads,
    ):
        super().__init__()

        self.q_norm = nn.LayerNorm(
            dim
        )

        self.kv_norm = nn.LayerNorm(
            dim
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True,
            dropout=0.05,
        )

        self.ff_norm = nn.LayerNorm(
            dim
        )

        self.ff = nn.Sequential(
            nn.Linear(
                dim,
                dim * 4,
            ),
            nn.GELU(),
            nn.Dropout(
                0.05
            ),
            nn.Linear(
                dim * 4,
                dim,
            ),
        )

    def forward(
        self,
        queries,
        visual_tokens,
    ):
        q = self.q_norm(
            queries
        )

        kv = self.kv_norm(
            visual_tokens
        )

        attn_out, _ = self.attn(
            q,
            kv,
            kv,
            need_weights=False,
        )

        x = (
            queries
            + attn_out
        )

        x = (
            x
            + self.ff(
                self.ff_norm(
                    x
                )
            )
        )

        return x

class FinalVisualBridge(nn.Module):
    def __init__(
        self,
        highres_dim,
        text_dim,
        latent_dim,
    ):
        super().__init__()

        self.patch_proj = nn.Sequential(
            nn.LayerNorm(
                highres_dim
            ),
            nn.Linear(
                highres_dim,
                VISUAL_HIDDEN,
            ),
        )

        self.instruction_proj = nn.Linear(
            text_dim,
            VISUAL_HIDDEN,
        )

        # Gaussian semantics:
        # mu(256), regression(3), dominance logits(3),
        # review logit(1), latent uncertainty(1)
        self.gaussian_dim = (
            latent_dim
            + 3
            + 3
            + 1
            + 1
        )

        self.gaussian_proj = nn.Sequential(
            nn.LayerNorm(
                self.gaussian_dim
            ),
            nn.Linear(
                self.gaussian_dim,
                VISUAL_HIDDEN,
            ),
            nn.GELU(),
        )

        self.query_tokens = nn.Parameter(
            torch.randn(
                1,
                N_QUERIES,
                VISUAL_HIDDEN,
            )
            * 0.02
        )

        self.blocks = nn.ModuleList([
            CrossAttentionBlock(
                VISUAL_HIDDEN,
                N_HEADS,
            )
            for _ in range(
                N_LAYERS
            )
        ])

        self.final_norm = nn.LayerNorm(
            VISUAL_HIDDEN
        )

        self.extent_head = nn.Linear(
            VISUAL_HIDDEN,
            3,
        )

        self.uncertainty_head = nn.Linear(
            VISUAL_HIDDEN,
            3,
        )

        self.review_head = nn.Linear(
            VISUAL_HIDDEN,
            2,
        )

        self.dominance_head = nn.Linear(
            VISUAL_HIDDEN,
            3,
        )

        # Continuous auxiliary predictions.
        self.regression_head = nn.Sequential(
            nn.Linear(
                VISUAL_HIDDEN,
                128,
            ),
            nn.GELU(),
            nn.Linear(
                128,
                3,
            ),
            nn.Sigmoid(),
        )

        # Tokens for FLAN-T5 planner.
        self.to_text_tokens = nn.Sequential(
            nn.Linear(
                VISUAL_HIDDEN,
                text_dim,
            ),
            nn.LayerNorm(
                text_dim
            ),
        )

        # Compact learned semantic token from visual heads.
        # 3 + 3 + 2 + 3 + 3 continuous = 14.
        self.semantic_to_text = nn.Sequential(
            nn.Linear(
                14,
                text_dim,
            ),
            nn.Tanh(),
        )

    def gaussian_vector(
        self,
        gout,
    ):
        return torch.cat(
            [
                gout["mu"].float(),
                gout["regression"].float(),
                gout["dominance_logits"].float(),
                gout["review_logit"].float().unsqueeze(-1),
                gout["latent_uncertainty"].float().unsqueeze(-1),
            ],
            dim=-1,
        )

    def forward(
        self,
        patch_tokens,
        instruction_context,
        gout,
    ):
        v = self.patch_proj(
            patch_tokens.float()
        )

        ic = self.instruction_proj(
            instruction_context.float()
        )

        gv = self.gaussian_proj(
            self.gaussian_vector(
                gout
            )
        )

        q = self.query_tokens.expand(
            patch_tokens.shape[0],
            -1,
            -1,
        )

        q = (
            q
            + ic.unsqueeze(1)
            + gv.unsqueeze(1)
        )

        for block in self.blocks:
            q = block(
                q,
                v,
            )

        q = self.final_norm(
            q
        )

        pooled = q.mean(
            dim=1
        )

        extent_logits = self.extent_head(
            pooled
        )

        uncertainty_logits = self.uncertainty_head(
            pooled
        )

        review_logits = self.review_head(
            pooled
        )

        dominance_logits = self.dominance_head(
            pooled
        )

        regression = self.regression_head(
            pooled
        )

        query_text_tokens = self.to_text_tokens(
            q
        )

        semantic_raw = torch.cat(
            [
                extent_logits,
                uncertainty_logits,
                review_logits,
                dominance_logits,
                regression,
            ],
            dim=-1,
        )

        semantic_token = self.semantic_to_text(
            semantic_raw
        ).unsqueeze(1)

        return {
            "queries": q,
            "pooled": pooled,
            "extent_logits": extent_logits,
            "uncertainty_logits": uncertainty_logits,
            "review_logits": review_logits,
            "dominance_logits": dominance_logits,
            "regression": regression,
            "text_tokens": torch.cat(
                [
                    semantic_token,
                    query_text_tokens,
                ],
                dim=1,
            ),
        }

visual_bridge = FinalVisualBridge(
    HIGHRES_DIM,
    TEXT_DIM,
    LATENT_DIM,
).to(DEVICE)

print("=" * 112)
print("P20-C09 — VISUAL BRIDGE")
print("=" * 112)
print(
    "Trainable params:",
    f"{sum(p.numel() for p in visual_bridge.parameters() if p.requires_grad):,}"
)
print("Query tokens:", N_QUERIES)
print("P20-C09 COMPLETE ✅")


P20-C09 — VISUAL BRIDGE
Trainable params: 7,847,454
Query tokens: 8
P20-C09 COMPLETE ✅


In [23]:
# ============================================================
# P20-C10 — VISUAL ALIGNMENT DATASET
# ============================================================
TASK_TO_ID = {
    "extent": 0,
    "uncertainty": 1,
    "review": 2,
    "dominance": 3,
}

def visual_task_label(
    scene_id,
    split,
    task,
):
    t = scene_targets(
        scene_id,
        split,
    )

    mapping = {
        "extent": "extent_class",
        "uncertainty": "uncertainty_class",
        "review": "review_class",
        "dominance": "dominance_class",
    }

    return int(
        t[
            mapping[
                task
            ]
        ]
    )

class VisualAlignmentDataset(Dataset):
    def __init__(
        self,
        scene_ids,
        split,
    ):
        self.rows = [
            (
                sid,
                task,
            )
            for sid in scene_ids
            for task in TASK_TO_ID
        ]

        self.split = split

    def __len__(self):
        return len(
            self.rows
        )

    def __getitem__(
        self,
        idx,
    ):
        sid, task = self.rows[
            idx
        ]

        h = HIGHRES_CACHE[
            sid
        ]

        g = GAUSSIAN_CACHE[
            sid
        ]

        return {
            "scene_id": sid,
            "task": task,
            "task_id": TASK_TO_ID[
                task
            ],
            "patch": h[
                "patch12"
            ].float(),
            "instruction_context": TASK_CONTEXT[
                task
            ].clone(),
            "gaussian": {
                k: v.clone()
                for k, v in g.items()
            },
            "label": visual_task_label(
                sid,
                self.split,
                task,
            ),
            "continuous": torch.tensor(
                scene_targets(
                    sid,
                    self.split,
                )["continuous"],
                dtype=torch.float32,
            ),
        }

def visual_collate(rows):
    gaussian_keys = list(
        rows[0]["gaussian"].keys()
    )

    return {
        "scene_ids": [
            x["scene_id"]
            for x in rows
        ],
        "tasks": [
            x["task"]
            for x in rows
        ],
        "task_ids": torch.tensor(
            [
                x["task_id"]
                for x in rows
            ],
            dtype=torch.long,
        ),
        "patch": torch.stack(
            [
                x["patch"]
                for x in rows
            ]
        ),
        "instruction_context": torch.stack(
            [
                x["instruction_context"]
                for x in rows
            ]
        ),
        "gaussian": {
            k: torch.stack(
                [
                    x["gaussian"][k]
                    for x in rows
                ]
            )
            for k in gaussian_keys
        },
        "labels": torch.tensor(
            [
                x["label"]
                for x in rows
            ],
            dtype=torch.long,
        ),
        "continuous": torch.stack(
            [
                x["continuous"]
                for x in rows
            ]
        ),
    }

visual_train_ds = VisualAlignmentDataset(
    train_scene_ids,
    "train",
)

visual_val_ds = VisualAlignmentDataset(
    val_scene_ids,
    "val",
)

# Balanced sampling across TASK+CLASS, following the principle
# of balancing heterogeneous instruction tasks.
group_counts = Counter()

for sid, task in visual_train_ds.rows:
    label = visual_task_label(
        sid,
        "train",
        task,
    )

    group_counts[
        (
            task,
            label,
        )
    ] += 1

sample_weights = []

for sid, task in visual_train_ds.rows:
    label = visual_task_label(
        sid,
        "train",
        task,
    )

    sample_weights.append(
        1.0
        / group_counts[
            (
                task,
                label,
            )
        ]
    )

visual_sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights,
        dtype=torch.double,
    ),
    num_samples=len(
        visual_train_ds
    ),
    replacement=True,
)

visual_train_loader = DataLoader(
    visual_train_ds,
    batch_size=64,
    sampler=visual_sampler,
    num_workers=0,
    collate_fn=visual_collate,
)

visual_val_loader = DataLoader(
    visual_val_ds,
    batch_size=136,
    shuffle=False,
    num_workers=0,
    collate_fn=visual_collate,
)

print("=" * 112)
print("P20-C10 — VISUAL ALIGNMENT DATA")
print("=" * 112)
print("Train rows:", len(visual_train_ds))
print("Val rows  :", len(visual_val_ds))
print("Train task-class counts:")
for k, v in sorted(group_counts.items()):
    print(k, ":", v)
print("P20-C10 COMPLETE ✅")


P20-C10 — VISUAL ALIGNMENT DATA
Train rows: 1256
Val rows  : 136
Train task-class counts:
('dominance', 0) : 269
('dominance', 1) : 42
('dominance', 2) : 3
('extent', 0) : 105
('extent', 1) : 104
('extent', 2) : 105
('review', 0) : 236
('review', 1) : 78
('uncertainty', 0) : 105
('uncertainty', 1) : 104
('uncertainty', 2) : 105
P20-C10 COMPLETE ✅


In [24]:
# ============================================================
# P20-C11 — VISUAL ALIGNMENT TRAINING
# Fixed protocol; best epoch chosen by validation CORE balanced
# accuracy (extent + uncertainty only).
# ============================================================
VISUAL_EPOCHS = 60

visual_optimizer = torch.optim.AdamW(
    visual_bridge.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

def move_gaussian_dict(
    gdict,
):
    return {
        k: v.to(
            DEVICE
        )
        for k, v in gdict.items()
    }

def logits_for_task(
    out,
    task,
):
    return {
        "extent": out["extent_logits"],
        "uncertainty": out["uncertainty_logits"],
        "review": out["review_logits"],
        "dominance": out["dominance_logits"],
    }[
        task
    ]

def batch_visual_loss(
    batch,
):
    patch = batch[
        "patch"
    ].to(DEVICE)

    context = batch[
        "instruction_context"
    ].to(DEVICE)

    labels = batch[
        "labels"
    ].to(DEVICE)

    continuous = batch[
        "continuous"
    ].to(DEVICE)

    gout = move_gaussian_dict(
        batch["gaussian"]
    )

    out = visual_bridge(
        patch,
        context,
        gout,
    )

    total_cls = patch.new_tensor(
        0.0
    )

    contributing = 0

    task_ids = batch[
        "task_ids"
    ].to(DEVICE)

    for task, tid in TASK_TO_ID.items():
        mask = (
            task_ids
            == tid
        )

        if not mask.any():
            continue

        logits = logits_for_task(
            out,
            task,
        )[
            mask
        ]

        y = labels[
            mask
        ]

        total_cls = (
            total_cls
            + F.cross_entropy(
                logits,
                y,
            )
        )

        contributing += 1

    total_cls = (
        total_cls
        / max(
            contributing,
            1,
        )
    )

    # Continuous supervision is visual-only and complements
    # discrete descriptive classes.
    reg_loss = F.mse_loss(
        out["regression"],
        continuous,
    )

    total = (
        total_cls
        + 0.30
        * reg_loss
    )

    return (
        total,
        total_cls,
        reg_loss,
        out,
    )

def balanced_accuracy(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    recalls = []

    for c in sorted(
        np.unique(
            y_true
        )
    ):
        mask = (
            y_true
            == c
        )

        recalls.append(
            float(
                np.mean(
                    y_pred[
                        mask
                    ]
                    == c
                )
            )
        )

    return float(
        np.mean(
            recalls
        )
    )

@torch.inference_mode()
def evaluate_visual_bridge(
    mode="full",
):
    visual_bridge.eval()

    rows = []

    for batch in visual_val_loader:
        scene_ids = batch[
            "scene_ids"
        ]

        tasks = batch[
            "tasks"
        ]

        contexts = batch[
            "instruction_context"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].numpy()

        if mode == "full":
            patch = batch[
                "patch"
            ].to(DEVICE)

            gout = move_gaussian_dict(
                batch[
                    "gaussian"
                ]
            )

        elif mode == "zero":
            patch = torch.zeros_like(
                batch["patch"]
            ).to(DEVICE)

            gout = {
                k: torch.zeros_like(
                    v
                ).to(DEVICE)
                for k, v in batch[
                    "gaussian"
                ].items()
            }

        elif mode == "shuffled":
            shuffled_patches = []
            shuffled_g = defaultdict(
                list
            )

            for sid in scene_ids:
                ssid = VAL_DERANGEMENT[
                    sid
                ]

                shuffled_patches.append(
                    HIGHRES_CACHE[
                        ssid
                    ][
                        "patch12"
                    ].float()
                )

                for k, v in GAUSSIAN_CACHE[
                    ssid
                ].items():
                    shuffled_g[
                        k
                    ].append(
                        v
                    )

            patch = torch.stack(
                shuffled_patches
            ).to(DEVICE)

            gout = {
                k: torch.stack(
                    vals
                ).to(DEVICE)
                for k, vals in shuffled_g.items()
            }

        else:
            raise ValueError(
                mode
            )

        out = visual_bridge(
            patch,
            contexts,
            gout,
        )

        task_ids = batch[
            "task_ids"
        ].numpy()

        for i, (
            sid,
            task,
            y,
        ) in enumerate(
            zip(
                scene_ids,
                tasks,
                labels,
            )
        ):
            logits = logits_for_task(
                out,
                task,
            )[
                i
            ]

            pred = int(
                logits.argmax().cpu()
            )

            rows.append({
                "scene_id": sid,
                "task": task,
                "target": int(y),
                "prediction": pred,
            })

    df = pd.DataFrame(
        rows
    )

    per_task = []

    for task in TASK_TO_ID:
        d = df[
            df["task"]
            == task
        ]

        per_task.append({
            "task": task,
            "accuracy": float(
                np.mean(
                    d["target"]
                    == d["prediction"]
                )
            ),
            "balanced_accuracy": balanced_accuracy(
                d["target"],
                d["prediction"],
            ),
        })

    per_task = pd.DataFrame(
        per_task
    )

    core = per_task[
        per_task["task"].isin(
            [
                "extent",
                "uncertainty",
            ]
        )
    ]

    core_balanced = float(
        core[
            "balanced_accuracy"
        ].mean()
    )

    return (
        df,
        per_task,
        core_balanced,
    )

best_core = -1.0
best_visual_epoch = None

VISUAL_BEST_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_10_visual_bridge.pt"
)

visual_history = []

for epoch in range(
    1,
    VISUAL_EPOCHS + 1,
):
    visual_bridge.train()

    running = []

    for batch in visual_train_loader:
        visual_optimizer.zero_grad(
            set_to_none=True
        )

        loss, cls_loss, reg_loss, _ = batch_visual_loss(
            batch
        )

        assert torch.isfinite(
            loss
        ).all()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            visual_bridge.parameters(),
            1.0,
        )

        visual_optimizer.step()

        running.append(
            float(
                loss.detach().cpu()
            )
        )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == VISUAL_EPOCHS
    ):
        _, per_task, core = evaluate_visual_bridge(
            "full"
        )

        visual_history.append({
            "epoch": epoch,
            "train_loss": float(
                np.mean(
                    running
                )
            ),
            "core_balanced_accuracy": core,
            "per_task": per_task.to_dict(
                orient="records"
            ),
        })

        print(
            f"Epoch {epoch:02d}/{VISUAL_EPOCHS} "
            f"train={np.mean(running):.4f} "
            f"core_bal_acc={core:.4f}"
        )

        if core > best_core:
            best_core = core
            best_visual_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "core_balanced_accuracy": core,
                    "state_dict": {
                        k: v.detach().cpu()
                        for k, v in visual_bridge.state_dict().items()
                    },
                },
                VISUAL_BEST_PATH,
            )

best_visual = torch.load(
    VISUAL_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

visual_bridge.load_state_dict(
    best_visual["state_dict"],
    strict=True,
)

visual_bridge = visual_bridge.to(
    DEVICE
).eval()

for p in visual_bridge.parameters():
    p.requires_grad = False

print("=" * 112)
print("P20-C11 — VISUAL ALIGNMENT COMPLETE")
print("=" * 112)
print("Best epoch:", best_visual_epoch)
print("Best core balanced accuracy:", best_core)
print("Visual bridge frozen for planner training.")
print("P20-C11 COMPLETE ✅")


Epoch 01/60 train=1.0293 core_bal_acc=0.6575
Epoch 05/60 train=0.1919 core_bal_acc=0.7387
Epoch 10/60 train=0.0486 core_bal_acc=0.6314
Epoch 15/60 train=0.0556 core_bal_acc=0.6549
Epoch 20/60 train=0.0060 core_bal_acc=0.6757
Epoch 25/60 train=0.0353 core_bal_acc=0.6353
Epoch 30/60 train=0.0102 core_bal_acc=0.7171
Epoch 35/60 train=0.0015 core_bal_acc=0.6924
Epoch 40/60 train=0.0006 core_bal_acc=0.6784
Epoch 45/60 train=0.0004 core_bal_acc=0.6784
Epoch 50/60 train=0.0004 core_bal_acc=0.6784
Epoch 55/60 train=0.0003 core_bal_acc=0.6784
Epoch 60/60 train=0.0003 core_bal_acc=0.6784
P20-C11 — VISUAL ALIGNMENT COMPLETE
Best epoch: 5
Best core balanced accuracy: 0.7387254901960785
Visual bridge frozen for planner training.
P20-C11 COMPLETE ✅


In [25]:
# ============================================================
# P20-C12 — TRUE VISUAL ABLATION
# ============================================================
full_visual_df, full_visual_task, full_core = (
    evaluate_visual_bridge(
        "full"
    )
)

zero_visual_df, zero_visual_task, zero_core = (
    evaluate_visual_bridge(
        "zero"
    )
)

shuffle_visual_df, shuffle_visual_task, shuffle_core = (
    evaluate_visual_bridge(
        "shuffled"
    )
)

print("=" * 112)
print("P20-C12 — TRUE VISUAL ABLATION")
print("=" * 112)

summary = pd.DataFrame([
    {
        "mode": "full",
        "core_balanced_accuracy": full_core,
    },
    {
        "mode": "zero",
        "core_balanced_accuracy": zero_core,
    },
    {
        "mode": "true_scene_shuffled",
        "core_balanced_accuracy": shuffle_core,
    },
])

display(
    summary
)

print("\nFULL per-task:")
display(
    full_visual_task
)

print("\nZERO per-task:")
display(
    zero_visual_task
)

print("\nTRUE SHUFFLED per-task:")
display(
    shuffle_visual_task
)

print()
print(
    "Full-zero gap   :",
    full_core
    - zero_core
)

print(
    "Full-shuffle gap:",
    full_core
    - shuffle_core
)

print("P20-C12 COMPLETE ✅")


P20-C12 — TRUE VISUAL ABLATION


,mode,core_balanced_accuracy
0,full,0.738725
1,zero,0.333333
2,true_scene_shuffled,0.293651



FULL per-task:


,task,accuracy,balanced_accuracy
0,extent,0.794118,0.730392
1,uncertainty,0.764706,0.747059
2,review,0.823529,0.857143
3,dominance,0.882353,0.500000



ZERO per-task:


,task,accuracy,balanced_accuracy
0,extent,0.235294,0.333333
1,uncertainty,0.294118,0.333333
2,review,0.617647,0.500000
3,dominance,0.882353,0.500000



TRUE SHUFFLED per-task:


,task,accuracy,balanced_accuracy
0,extent,0.323529,0.267974
1,uncertainty,0.352941,0.319328
2,review,0.352941,0.358974
3,dominance,0.882353,0.500000



Full-zero gap   : 0.40539215686274516
Full-shuffle gap: 0.44507469654528486
P20-C12 COMPLETE ✅


## Planner stage

The local VLM no longer generates raw scientific numbers.

Instead, it predicts one item from a **finite response-plan vocabulary**.

Examples:

```text
PLAN_EXTENT
PLAN_UNCERTAINTY
PLAN_SENSOR_ATTRIBUTION
PLAN_REVIEW
REFUSE_PHYSICAL_CAUSE
VISUAL_EXTENT_LOW
VISUAL_UNCERTAINTY_HIGH
```

The deterministic renderer then reads the verified packet and inserts exact values.

This design makes factual ownership explicit:

```text
VLM = intent + visual semantics + plan
GeoEngine = numerical facts
Renderer/verifier = final grounded statement
```


In [26]:
# ============================================================
# P20-C13 — PLAN VOCABULARY + FINAL TRAINING EXAMPLES
# ============================================================
GROUNDED_TASK_TO_PLAN = {
    "extent": "PLAN_EXTENT",
    "uncertainty": "PLAN_UNCERTAINTY",
    "sensor_attribution": "PLAN_SENSOR_ATTRIBUTION",
    "temporary_sensor_attribution": "PLAN_TEMP_ATTRIBUTION",
    "spatial_geometry": "PLAN_SPATIAL_GEOMETRY",
    "integrated_summary": "PLAN_INTEGRATED_SUMMARY",
    "review_policy": "PLAN_REVIEW",
    "cause_refusal": "REFUSE_PHYSICAL_CAUSE",
    "impact_refusal": "REFUSE_HUMAN_IMPACT",
    "forecast_refusal": "REFUSE_FUTURE_FORECAST",
}

EXTENT_NAMES = [
    "LOW",
    "MEDIUM",
    "HIGH",
]

UNCERTAINTY_NAMES = [
    "LOW",
    "MEDIUM",
    "HIGH",
]

REVIEW_NAMES = [
    "NO",
    "YES",
]

DOMINANCE_NAMES = [
    "OPTICAL",
    "BALANCED",
    "SAR",
]

def visual_plan(
    scene_id,
    split,
    task,
):
    t = scene_targets(
        scene_id,
        split,
    )

    if task == "visual_extent":
        return (
            "VISUAL_EXTENT_"
            + EXTENT_NAMES[
                t["extent_class"]
            ]
        )

    if task == "visual_uncertainty":
        return (
            "VISUAL_UNCERTAINTY_"
            + UNCERTAINTY_NAMES[
                t["uncertainty_class"]
            ]
        )

    if task == "visual_review":
        return (
            "VISUAL_REVIEW_"
            + REVIEW_NAMES[
                t["review_class"]
            ]
        )

    if task == "visual_dominance":
        return (
            "VISUAL_DOMINANCE_"
            + DOMINANCE_NAMES[
                t["dominance_class"]
            ]
        )

    raise KeyError(
        task
    )

VISUAL_PROMPTS = {
    "visual_extent": (
        "Using the image evidence, classify the temporary-flood extent descriptor."
    ),
    "visual_uncertainty": (
        "Using the image evidence, classify the FloodShift uncertainty descriptor."
    ),
    "visual_review": (
        "Using the image evidence, determine whether the fixed review policy flags the scene."
    ),
    "visual_dominance": (
        "Using the image evidence, classify the sensor-dependence dominance."
    ),
}

def planner_examples(
    base_examples,
    scene_ids,
    split,
):
    rows = []

    for ex in base_examples:
        rows.append({
            "id": ex["id"],
            "scene_id": ex["scene_id"],
            "instruction": ex["instruction"],
            "plan": GROUNDED_TASK_TO_PLAN[
                ex["task"]
            ],
            "task": ex["task"],
            "is_visual_only": False,
        })

    for sid in scene_ids:
        for task, prompt in VISUAL_PROMPTS.items():
            rows.append({
                "id": f"{sid}__{task}",
                "scene_id": sid,
                "instruction": prompt,
                "plan": visual_plan(
                    sid,
                    split,
                    task,
                ),
                "task": task,
                "is_visual_only": True,
            })

    return rows

train_plan_rows = planner_examples(
    train_examples,
    train_scene_ids,
    "train",
)

val_plan_rows = planner_examples(
    val_examples,
    val_scene_ids,
    "val",
)

PLAN_VOCAB = sorted({
    x["plan"]
    for x in train_plan_rows
    + val_plan_rows
})

PLAN_TO_ID = {
    p: i
    for i, p in enumerate(
        PLAN_VOCAB
    )
}

ID_TO_PLAN = {
    i: p
    for p, i in PLAN_TO_ID.items()
}

print("=" * 112)
print("P20-C13 — PLAN VOCABULARY")
print("=" * 112)
print("Train planner rows:", len(train_plan_rows))
print("Val planner rows  :", len(val_plan_rows))
print("Plan classes      :", len(PLAN_VOCAB))
print()
for p in PLAN_VOCAB:
    print(
        f"{PLAN_TO_ID[p]:02d}",
        p,
    )
print("P20-C13 COMPLETE ✅")


P20-C13 — PLAN VOCABULARY
Train planner rows: 4396
Val planner rows  : 476
Plan classes      : 21

00 PLAN_EXTENT
01 PLAN_INTEGRATED_SUMMARY
02 PLAN_REVIEW
03 PLAN_SENSOR_ATTRIBUTION
04 PLAN_SPATIAL_GEOMETRY
05 PLAN_TEMP_ATTRIBUTION
06 PLAN_UNCERTAINTY
07 REFUSE_FUTURE_FORECAST
08 REFUSE_HUMAN_IMPACT
09 REFUSE_PHYSICAL_CAUSE
10 VISUAL_DOMINANCE_BALANCED
11 VISUAL_DOMINANCE_OPTICAL
12 VISUAL_DOMINANCE_SAR
13 VISUAL_EXTENT_HIGH
14 VISUAL_EXTENT_LOW
15 VISUAL_EXTENT_MEDIUM
16 VISUAL_REVIEW_NO
17 VISUAL_REVIEW_YES
18 VISUAL_UNCERTAINTY_HIGH
19 VISUAL_UNCERTAINTY_LOW
20 VISUAL_UNCERTAINTY_MEDIUM
P20-C13 COMPLETE ✅


In [28]:
# ============================================================
# P20-C14 — PLANNER DATASET
# FIXED: raw planner rows contain "plan", not "plan_id"
# ============================================================

MAX_TEXT_LENGTH = 128


class PlannerDataset(Dataset):
    def __init__(
        self,
        rows,
        split,
    ):
        self.rows = rows
        self.split = split

        self.derangement = (
            TRAIN_DERANGEMENT
            if split == "train"
            else VAL_DERANGEMENT
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(
        self,
        idx,
    ):
        row = self.rows[idx]

        sid = row["scene_id"]

        ssid = self.derangement[sid]

        # Convert symbolic plan -> integer class here
        plan_id = PLAN_TO_ID[
            row["plan"]
        ]

        return {
            **row,

            "plan_id": plan_id,

            "patch": HIGHRES_CACHE[
                sid
            ]["patch12"].float(),

            "gaussian": {
                k: v.clone()
                for k, v in GAUSSIAN_CACHE[
                    sid
                ].items()
            },

            "shuffled_patch": HIGHRES_CACHE[
                ssid
            ]["patch12"].float(),

            "shuffled_gaussian": {
                k: v.clone()
                for k, v in GAUSSIAN_CACHE[
                    ssid
                ].items()
            },
        }


def planner_collate(rows):

    text = tokenizer(
        [
            x["instruction"]
            for x in rows
        ],
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt",
    )

    gkeys = list(
        rows[0][
            "gaussian"
        ].keys()
    )

    return {
        "ids": [
            x["id"]
            for x in rows
        ],

        "scene_ids": [
            x["scene_id"]
            for x in rows
        ],

        "tasks": [
            x["task"]
            for x in rows
        ],

        "plans": [
            x["plan"]
            for x in rows
        ],

        "is_visual_only": torch.tensor(
            [
                x["is_visual_only"]
                for x in rows
            ],
            dtype=torch.bool,
        ),

        "plan_ids": torch.tensor(
            [
                x["plan_id"]
                for x in rows
            ],
            dtype=torch.long,
        ),

        "input_ids": text[
            "input_ids"
        ],

        "attention_mask": text[
            "attention_mask"
        ],

        "patch": torch.stack(
            [
                x["patch"]
                for x in rows
            ]
        ),

        "gaussian": {
            k: torch.stack(
                [
                    x["gaussian"][k]
                    for x in rows
                ]
            )
            for k in gkeys
        },

        "shuffled_patch": torch.stack(
            [
                x["shuffled_patch"]
                for x in rows
            ]
        ),

        "shuffled_gaussian": {
            k: torch.stack(
                [
                    x["shuffled_gaussian"][k]
                    for x in rows
                ]
            )
            for k in gkeys
        },
    }


# ============================================================
# CREATE DATASETS
# ============================================================

train_plan_ds = PlannerDataset(
    train_plan_rows,
    "train",
)

val_plan_ds = PlannerDataset(
    val_plan_rows,
    "val",
)


# ============================================================
# BALANCE PLAN CLASSES
#
# IMPORTANT FIX:
# train_plan_rows contains "plan", NOT "plan_id".
# Therefore use PLAN_TO_ID[x["plan"]].
# ============================================================

plan_counts = Counter(
    PLAN_TO_ID[
        x["plan"]
    ]
    for x in train_plan_rows
)


planner_weights = [
    1.0
    / plan_counts[
        PLAN_TO_ID[
            x["plan"]
        ]
    ]
    for x in train_plan_rows
]


planner_sampler = WeightedRandomSampler(
    weights=torch.tensor(
        planner_weights,
        dtype=torch.double,
    ),

    num_samples=len(
        train_plan_ds
    ),

    replacement=True,
)


# ============================================================
# DATA LOADERS
# ============================================================

train_plan_loader = DataLoader(
    train_plan_ds,

    batch_size=8,

    sampler=planner_sampler,

    num_workers=0,

    collate_fn=planner_collate,
)


val_plan_loader = DataLoader(
    val_plan_ds,

    batch_size=16,

    shuffle=False,

    num_workers=0,

    collate_fn=planner_collate,
)


# ============================================================
# SANITY CHECK
# ============================================================

sample_batch = next(
    iter(
        train_plan_loader
    )
)

assert (
    sample_batch[
        "plan_ids"
    ].dtype
    == torch.long
)

assert (
    sample_batch[
        "plan_ids"
    ].min()
    >= 0
)

assert (
    sample_batch[
        "plan_ids"
    ].max()
    < len(
        PLAN_VOCAB
    )
)


# ============================================================
# REPORT
# ============================================================

print("=" * 112)

print(
    "P20-C14 — PLANNER DATA"
)

print("=" * 112)

print(
    "Train rows:",
    len(
        train_plan_ds
    ),
)

print(
    "Val rows  :",
    len(
        val_plan_ds
    ),
)

print(
    "Plan classes:",
    len(
        PLAN_VOCAB
    ),
)

print()

print(
    "Plan counts:"
)

for pid, count in sorted(
    plan_counts.items()
):
    print(
        f"{pid:02d}",
        ID_TO_PLAN[
            pid
        ],
        ":",
        count,
    )


print()

print(
    "Sample batch shape:",
    tuple(
        sample_batch[
            "plan_ids"
        ].shape
    ),
)

print(
    "Sample plan IDs:",
    sample_batch[
        "plan_ids"
    ].tolist(),
)

print()

print(
    "P20-C14 COMPLETE ✅"
)

P20-C14 — PLANNER DATA
Train rows: 4396
Val rows  : 476
Plan classes: 21

Plan counts:
00 PLAN_EXTENT : 314
01 PLAN_INTEGRATED_SUMMARY : 314
02 PLAN_REVIEW : 314
03 PLAN_SENSOR_ATTRIBUTION : 314
04 PLAN_SPATIAL_GEOMETRY : 314
05 PLAN_TEMP_ATTRIBUTION : 314
06 PLAN_UNCERTAINTY : 314
07 REFUSE_FUTURE_FORECAST : 314
08 REFUSE_HUMAN_IMPACT : 314
09 REFUSE_PHYSICAL_CAUSE : 314
10 VISUAL_DOMINANCE_BALANCED : 42
11 VISUAL_DOMINANCE_OPTICAL : 269
12 VISUAL_DOMINANCE_SAR : 3
13 VISUAL_EXTENT_HIGH : 105
14 VISUAL_EXTENT_LOW : 105
15 VISUAL_EXTENT_MEDIUM : 104
16 VISUAL_REVIEW_NO : 236
17 VISUAL_REVIEW_YES : 78
18 VISUAL_UNCERTAINTY_HIGH : 105
19 VISUAL_UNCERTAINTY_LOW : 105
20 VISUAL_UNCERTAINTY_MEDIUM : 104

Sample batch shape: (8,)
Sample plan IDs: [0, 6, 20, 20, 5, 6, 9, 11]

P20-C14 COMPLETE ✅


In [29]:
# ============================================================
# P20-C15 — LoRA FLAN-T5 ENCODER + PLAN CLASSIFIER
# ============================================================
# Recreate a trainable encoder from the same pretrained checkpoint.
del text_encoder
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_encoder = T5EncoderModel.from_pretrained(
    TEXT_ENCODER_NAME
)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q",
        "v",
    ],
    bias="none",
)

text_encoder = get_peft_model(
    base_encoder,
    lora_config,
).to(DEVICE)

class PlanHead(nn.Module):
    def __init__(
        self,
        d_model,
        n_plans,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(
                d_model
            ),
            nn.Linear(
                d_model,
                512,
            ),
            nn.GELU(),
            nn.Dropout(
                0.10
            ),
            nn.Linear(
                512,
                n_plans,
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.net(
            x
        )

plan_head = PlanHead(
    TEXT_DIM,
    len(
        PLAN_VOCAB
    ),
).to(DEVICE)

print("=" * 112)
print("P20-C15 — PLAN CLASSIFIER")
print("=" * 112)
print(
    "LoRA trainable:",
    f"{sum(p.numel() for p in text_encoder.parameters() if p.requires_grad):,}"
)
print(
    "Plan head:",
    f"{sum(p.numel() for p in plan_head.parameters() if p.requires_grad):,}"
)
print("Visual bridge: FROZEN")
print("P20-C15 COMPLETE ✅")


P20-C15 — PLAN CLASSIFIER
LoRA trainable: 294,912
Plan head: 406,037
Visual bridge: FROZEN
P20-C15 COMPLETE ✅


In [30]:
# ============================================================
# P20-C16 — MULTIMODAL PLANNER FORWARD
# ============================================================
def get_instruction_context_from_ids(
    input_ids,
    attention_mask,
):
    emb = text_encoder.get_input_embeddings()(
        input_ids
    )

    mask = attention_mask.unsqueeze(
        -1
    ).float()

    return (
        (
            emb
            * mask
        ).sum(
            dim=1
        )
        / mask.sum(
            dim=1
        ).clamp_min(
            1.0
        )
    )

@torch.no_grad()
def frozen_visual_tokens(
    patch,
    context,
    gout,
):
    visual_bridge.eval()

    out = visual_bridge(
        patch,
        context,
        gout,
    )

    return out[
        "text_tokens"
    ]

def planner_forward(
    batch,
    vision_mode="full",
):
    ids = batch[
        "input_ids"
    ].to(DEVICE)

    mask = batch[
        "attention_mask"
    ].to(DEVICE)

    context = get_instruction_context_from_ids(
        ids,
        mask,
    )

    if vision_mode == "full":
        patch = batch[
            "patch"
        ].to(DEVICE)

        gout = move_gaussian_dict(
            batch[
                "gaussian"
            ]
        )

    elif vision_mode == "zero":
        patch = torch.zeros_like(
            batch[
                "patch"
            ]
        ).to(DEVICE)

        gout = {
            k: torch.zeros_like(
                v
            ).to(DEVICE)
            for k, v in batch[
                "gaussian"
            ].items()
        }

    elif vision_mode == "shuffled":
        patch = batch[
            "shuffled_patch"
        ].to(DEVICE)

        gout = move_gaussian_dict(
            batch[
                "shuffled_gaussian"
            ]
        )

    else:
        raise ValueError(
            vision_mode
        )

    vis_tokens = frozen_visual_tokens(
        patch,
        context.detach(),
        gout,
    )

    text_emb = text_encoder.get_input_embeddings()(
        ids
    )

    embeds = torch.cat(
        [
            vis_tokens,
            text_emb,
        ],
        dim=1,
    )

    vis_mask = torch.ones(
        (
            ids.shape[0],
            vis_tokens.shape[1],
        ),
        dtype=mask.dtype,
        device=DEVICE,
    )

    mm_mask = torch.cat(
        [
            vis_mask,
            mask,
        ],
        dim=1,
    )

    out = text_encoder(
        inputs_embeds=embeds,
        attention_mask=mm_mask,
        return_dict=True,
    )

    hidden = out.last_hidden_state

    # Mean over the visual-token positions only.
    pooled = hidden[
        :,
        :vis_tokens.shape[1],
        :
    ].mean(
        dim=1
    )

    logits = plan_head(
        pooled
    )

    return logits

# Smoke test.
smoke = next(
    iter(
        train_plan_loader
    )
)

logits = planner_forward(
    smoke,
    "full",
)

loss = F.cross_entropy(
    logits,
    smoke[
        "plan_ids"
    ].to(DEVICE),
)

assert torch.isfinite(
    loss
).all()

print("=" * 112)
print("P20-C16 — FP32 SMOKE TEST")
print("=" * 112)
print("Logits:", tuple(logits.shape))
print("Loss  :", float(loss.detach().cpu()))
print("Finite: PASS")
print("P20-C16 COMPLETE ✅")


P20-C16 — FP32 SMOKE TEST
Logits: (8, 21)
Loss  : 3.1681790351867676
Finite: PASS
P20-C16 COMPLETE ✅


In [31]:
# ============================================================
# P20-C17 — FIXED PLANNER TRAINING
# ============================================================
PLANNER_EPOCHS = 5
GRAD_ACCUM = 4

planner_optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p
                for p in text_encoder.parameters()
                if p.requires_grad
            ],
            "lr": 1e-4,
        },
        {
            "params": plan_head.parameters(),
            "lr": 3e-4,
        },
    ],
    weight_decay=1e-4,
)

PLANNER_BEST_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_10_best_planner.pt"
)

@torch.inference_mode()
def evaluate_planner(
    mode="full",
):
    text_encoder.eval()
    plan_head.eval()

    rows = []

    for batch in val_plan_loader:
        logits = planner_forward(
            batch,
            mode,
        )

        pred = (
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )

        target = batch[
            "plan_ids"
        ].numpy()

        for i in range(
            len(
                target
            )
        ):
            rows.append({
                "id": batch["ids"][i],
                "scene_id": batch["scene_ids"][i],
                "task": batch["tasks"][i],
                "is_visual_only": bool(
                    batch["is_visual_only"][i]
                ),
                "target_id": int(
                    target[i]
                ),
                "pred_id": int(
                    pred[i]
                ),
                "target_plan": ID_TO_PLAN[
                    int(
                        target[i]
                    )
                ],
                "pred_plan": ID_TO_PLAN[
                    int(
                        pred[i]
                    )
                ],
            })

    df = pd.DataFrame(
        rows
    )

    grounded = df[
        ~df[
            "is_visual_only"
        ]
    ]

    visual = df[
        df[
            "is_visual_only"
        ]
    ]

    return {
        "df": df,
        "overall_accuracy": float(
            np.mean(
                df["target_id"]
                == df["pred_id"]
            )
        ),
        "grounded_accuracy": float(
            np.mean(
                grounded["target_id"]
                == grounded["pred_id"]
            )
        ),
        "visual_accuracy": float(
            np.mean(
                visual["target_id"]
                == visual["pred_id"]
            )
        ),
    }

best_grounded = -1.0
best_planner_epoch = None

planner_history = []

for epoch in range(
    1,
    PLANNER_EPOCHS + 1,
):
    text_encoder.train()
    plan_head.train()

    planner_optimizer.zero_grad(
        set_to_none=True
    )

    running = []

    for bi, batch in enumerate(
        train_plan_loader,
        start=1,
    ):
        logits = planner_forward(
            batch,
            "full",
        )

        target = batch[
            "plan_ids"
        ].to(DEVICE)

        loss = F.cross_entropy(
            logits,
            target,
        )

        assert torch.isfinite(
            loss
        ).all()

        (
            loss
            / GRAD_ACCUM
        ).backward()

        should_step = (
            bi % GRAD_ACCUM == 0
            or bi == len(
                train_plan_loader
            )
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in list(
                        text_encoder.parameters()
                    )
                    + list(
                        plan_head.parameters()
                    )
                    if p.requires_grad
                ],
                1.0,
            )

            planner_optimizer.step()

            planner_optimizer.zero_grad(
                set_to_none=True
            )

        running.append(
            float(
                loss.detach().cpu()
            )
        )

    vm = evaluate_planner(
        "full"
    )

    planner_history.append({
        "epoch": epoch,
        "train_loss": float(
            np.mean(
                running
            )
        ),
        "overall_accuracy": vm[
            "overall_accuracy"
        ],
        "grounded_accuracy": vm[
            "grounded_accuracy"
        ],
        "visual_accuracy": vm[
            "visual_accuracy"
        ],
    })

    print(
        f"Epoch {epoch}/{PLANNER_EPOCHS} "
        f"train={np.mean(running):.4f} "
        f"overall={vm['overall_accuracy']:.4f} "
        f"grounded={vm['grounded_accuracy']:.4f} "
        f"visual={vm['visual_accuracy']:.4f}"
    )

    if vm[
        "grounded_accuracy"
    ] > best_grounded:
        best_grounded = vm[
            "grounded_accuracy"
        ]

        best_planner_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "grounded_accuracy": best_grounded,
                "peft_state_dict": {
                    k: v.detach().cpu()
                    for k, v in get_peft_model_state_dict(
                        text_encoder
                    ).items()
                },
                "plan_head_state_dict": {
                    k: v.detach().cpu()
                    for k, v in plan_head.state_dict().items()
                },
            },
            PLANNER_BEST_PATH,
        )

best_planner = torch.load(
    PLANNER_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

set_peft_model_state_dict(
    text_encoder,
    best_planner[
        "peft_state_dict"
    ],
)

plan_head.load_state_dict(
    best_planner[
        "plan_head_state_dict"
    ],
    strict=True,
)

text_encoder = text_encoder.to(
    DEVICE
).eval()

plan_head = plan_head.to(
    DEVICE
).eval()

print("=" * 112)
print("P20-C17 — PLANNER COMPLETE")
print("=" * 112)
print("Best epoch:", best_planner_epoch)
print("Best grounded accuracy:", best_grounded)
print("P20-C17 COMPLETE ✅")


Epoch 1/5 train=1.1084 overall=0.9160 grounded=1.0000 visual=0.7059
Epoch 2/5 train=0.1398 overall=0.9223 grounded=1.0000 visual=0.7279
Epoch 3/5 train=0.1201 overall=0.9286 grounded=1.0000 visual=0.7500
Epoch 4/5 train=0.0945 overall=0.9307 grounded=1.0000 visual=0.7574
Epoch 5/5 train=0.0828 overall=0.9244 grounded=1.0000 visual=0.7353
P20-C17 — PLANNER COMPLETE
Best epoch: 1
Best grounded accuracy: 1.0
P20-C17 COMPLETE ✅


In [32]:
# ============================================================
# P20-C18 — FINAL PLANNER ABLATIONS
# ============================================================
planner_full = evaluate_planner(
    "full"
)

planner_zero = evaluate_planner(
    "zero"
)

planner_shuffle = evaluate_planner(
    "shuffled"
)

planner_ablation = pd.DataFrame([
    {
        "mode": "full",
        "overall_accuracy": planner_full[
            "overall_accuracy"
        ],
        "grounded_accuracy": planner_full[
            "grounded_accuracy"
        ],
        "visual_accuracy": planner_full[
            "visual_accuracy"
        ],
    },
    {
        "mode": "zero",
        "overall_accuracy": planner_zero[
            "overall_accuracy"
        ],
        "grounded_accuracy": planner_zero[
            "grounded_accuracy"
        ],
        "visual_accuracy": planner_zero[
            "visual_accuracy"
        ],
    },
    {
        "mode": "true_scene_shuffled",
        "overall_accuracy": planner_shuffle[
            "overall_accuracy"
        ],
        "grounded_accuracy": planner_shuffle[
            "grounded_accuracy"
        ],
        "visual_accuracy": planner_shuffle[
            "visual_accuracy"
        ],
    },
])

display(
    planner_ablation
)

print(
    "Visual full-zero gap:",
    planner_full[
        "visual_accuracy"
    ]
    - planner_zero[
        "visual_accuracy"
    ],
)

print(
    "Visual full-shuffle gap:",
    planner_full[
        "visual_accuracy"
    ]
    - planner_shuffle[
        "visual_accuracy"
    ],
)

print("P20-C18 COMPLETE ✅")


,mode,overall_accuracy,grounded_accuracy,visual_accuracy
0,full,0.915966,1.0,0.705882
1,zero,0.859244,1.0,0.507353
2,true_scene_shuffled,0.848739,1.0,0.470588


Visual full-zero gap: 0.19852941176470595
Visual full-shuffle gap: 0.23529411764705888
P20-C18 COMPLETE ✅


In [33]:
# ============================================================
# P20-C19 — DETERMINISTIC VERIFIED RENDERER
#
# No model-generated scientific numbers.
# Every number is inserted from the verified packet.
# ============================================================
def pct(x):
    return f"{100.0 * float(x):.2f}%"

def dec(x):
    return f"{float(x):.3f}"

def area(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.2f} km²"

def safe_get(
    d,
    *keys,
    default=None,
):
    cur = d

    for k in keys:
        if not isinstance(
            cur,
            dict,
        ):
            return default

        cur = cur.get(
            k,
            default,
        )

        if cur is default:
            return default

    return cur

def render_plan(
    scene_id,
    plan,
):
    p = packet_by_scene[
        scene_id
    ]

    seg = p[
        "segmentation"
    ]

    unc = p[
        "uncertainty"
    ]

    sq = p[
        "sensor_quality"
    ]

    att = p[
        "counterfactual_attribution"
    ]

    geo = p[
        "spatial_geometry"
    ]

    used = {}

    if plan == "PLAN_EXTENT":
        used = {
            "temp_fraction": float(
                seg["temporary_flood"]["fraction_of_coverage"]
            ),
            "temp_area_km2": seg[
                "temporary_flood"
            ][
                "area_km2"
            ],
            "perm_fraction": float(
                seg["permanent_water"]["fraction_of_coverage"]
            ),
        }

        text = (
            f"Temporary flooding covers {pct(used['temp_fraction'])} "
            f"of the sensor-covered area ({area(used['temp_area_km2'])}). "
            f"Permanent water covers {pct(used['perm_fraction'])}."
        )

    elif plan == "PLAN_UNCERTAINTY":
        used = {
            "entropy": float(
                unc["mean_entropy"]
            ),
            "review10": float(
                unc["review10_fraction"]
            ),
            "review20": float(
                unc["review20_fraction"]
            ),
            "pclear": float(
                sq["mean_pclear"]
            ),
        }

        text = (
            f"Mean normalized predictive entropy is {dec(used['entropy'])}. "
            f"The top review fractions are {pct(used['review10'])} "
            f"and {pct(used['review20'])}; mean P(clear) is "
            f"{dec(used['pclear'])}. These are predictive-reliability "
            f"indicators, not Bayesian epistemic uncertainty."
        )

    elif plan == "PLAN_SENSOR_ATTRIBUTION":
        used = {
            "optical_share": float(
                att["optical_share"]
            ),
            "sar_share": float(
                att["sar_share"]
            ),
            "flip_no_optical": float(
                att["flip_rate_no_optical"]
            ),
            "flip_no_sar": float(
                att["flip_rate_no_sar"]
            ),
        }

        dominance = str(
            att["dominance"]
        )

        text = (
            f"Counterfactual dependence is {dominance}. "
            f"Optical share is {pct(used['optical_share'])} and "
            f"SAR share is {pct(used['sar_share'])}. "
            f"Removing optical flips {pct(used['flip_no_optical'])} "
            f"of predictions, while removing SAR flips "
            f"{pct(used['flip_no_sar'])}. "
            f"These are model-dependence measures, not physical causality."
        )

    elif plan == "PLAN_TEMP_ATTRIBUTION":
        used = {
            "temp_js_optical": float(
                att["temporary_flood_js_optical"]
            ),
            "temp_js_sar": float(
                att["temporary_flood_js_sar"]
            ),
        }

        text = (
            f"For temporary-flood predictions, normalized JS dependence "
            f"is {dec(used['temp_js_optical'])} for optical removal and "
            f"{dec(used['temp_js_sar'])} for SAR removal. "
            f"These values describe model sensitivity, not physical cause."
        )

    elif plan == "PLAN_SPATIAL_GEOMETRY":
        components = safe_get(
            geo,
            "temporary_flood_connected_components_8conn",
            default=0,
        )

        largest_pixels = safe_get(
            geo,
            "largest_temporary_component_pixels",
            default=None,
        )

        largest_area = safe_get(
            geo,
            "largest_temporary_component_area_km2",
            default=None,
        )

        used = {
            "components": float(
                components
            ),
        }

        if largest_pixels is not None:
            used[
                "largest_component_pixels"
            ] = float(
                largest_pixels
            )

        if largest_area is not None:
            used[
                "largest_component_area_km2"
            ] = float(
                largest_area
            )

        bits = [
            f"The temporary-flood mask contains {int(components)} "
            f"raw 8-connected components."
        ]

        if largest_pixels is not None:
            bits.append(
                f"The largest component contains {int(largest_pixels)} pixels."
            )

        if largest_area is not None:
            bits.append(
                f"Its area is {area(largest_area)}."
            )

        bits.append(
            "Connected components are pixel-mask structures, not separate flood events."
        )

        text = " ".join(
            bits
        )

    elif plan == "PLAN_INTEGRATED_SUMMARY":
        used = {
            "temp_fraction": float(
                seg["temporary_flood"]["fraction_of_coverage"]
            ),
            "temp_area": seg[
                "temporary_flood"
            ][
                "area_km2"
            ],
            "entropy": float(
                unc["mean_entropy"]
            ),
            "pclear": float(
                sq["mean_pclear"]
            ),
            "optical_share": float(
                att["optical_share"]
            ),
            "sar_share": float(
                att["sar_share"]
            ),
        }

        text = (
            f"FloodShift predicts temporary flooding over "
            f"{pct(used['temp_fraction'])} of covered area "
            f"({area(used['temp_area'])}). Mean predictive entropy is "
            f"{dec(used['entropy'])}, mean P(clear) is "
            f"{dec(used['pclear'])}, and counterfactual dependence is "
            f"{pct(used['optical_share'])} optical versus "
            f"{pct(used['sar_share'])} SAR."
        )

    elif plan == "PLAN_REVIEW":
        review = (
            val_review.get(
                scene_id,
                train_review.get(
                    scene_id,
                    0,
                ),
            )
        )

        used = {
            "review10": float(
                unc["review10_fraction"]
            ),
            "high_fragility": float(
                att["high_fragility_tile_fraction"]
            ),
        }

        if review:
            prefix = (
                "Yes. The fixed FloodShift policy flags this scene for review."
            )
        else:
            prefix = (
                "No. The fixed FloodShift policy does not flag this scene for review."
            )

        text = (
            prefix
            + f" Review10 fraction is {pct(used['review10'])} "
            + f"and high-fragility tile fraction is "
            + f"{pct(used['high_fragility'])}."
        )

    elif plan == "REFUSE_PHYSICAL_CAUSE":
        text = (
            "FloodShift does not establish the physical cause of a flood. "
            "Counterfactual sensor attribution measures model dependence, "
            "not physical causality."
        )

    elif plan == "REFUSE_HUMAN_IMPACT":
        text = (
            "FloodShift does not contain verified population, casualty, "
            "property-damage, or human-impact data, so it cannot determine "
            "those impacts from this evidence."
        )

    elif plan == "REFUSE_FUTURE_FORECAST":
        text = (
            "FloodShift's current evidence packet does not provide a future "
            "flood-propagation forecast. A forecast would require external "
            "temporal or hydrologic evidence."
        )

    elif plan.startswith(
        "VISUAL_"
    ):
        # Visual descriptors contain no scientific numerical claims.
        text = plan

    else:
        raise KeyError(
            f"Unknown plan: {plan}"
        )

    return {
        "text": text,
        "used_facts": used,
    }

print("P20-C19 COMPLETE ✅")


P20-C19 COMPLETE ✅


In [35]:
# ============================================================
# P20-C19 — DETERMINISTIC VERIFIED RENDERER
#
# No model-generated scientific numbers.
# Every numerical value is inserted from the verified GeoEngine
# evidence packet.
#
# Missing evidence -> explicit deterministic abstention.
# ============================================================


# ------------------------------------------------------------
# Formatting helpers
# ------------------------------------------------------------

def pct(x):
    if x is None:
        return "unavailable"

    return f"{100.0 * float(x):.2f}%"


def dec(x):
    if x is None:
        return "unavailable"

    return f"{float(x):.3f}"


def area(x):
    if x is None:
        return "unavailable"

    return f"{float(x):.2f} km²"


def safe_get(
    d,
    *keys,
    default=None,
):
    """
    Safely traverse nested dictionaries.
    """

    cur = d

    for k in keys:

        if not isinstance(
            cur,
            dict,
        ):
            return default

        if k not in cur:
            return default

        cur = cur[k]

    return cur


def finite_number_or_none(x):
    """
    Convert a value to float only when it exists
    and is finite.

    None / NaN / invalid strings -> None.
    """

    if x is None:
        return None

    try:
        value = float(x)

    except (
        TypeError,
        ValueError,
    ):
        return None

    if not np.isfinite(value):
        return None

    return value


# ============================================================
# MAIN VERIFIED RENDERER
# ============================================================

def render_plan(
    scene_id,
    plan,
):

    # --------------------------------------------------------
    # Get verified GeoEngine packet
    # --------------------------------------------------------

    p = packet_by_scene[
        scene_id
    ]

    seg = p[
        "segmentation"
    ]

    unc = p[
        "uncertainty"
    ]

    sq = p[
        "sensor_quality"
    ]

    att = p[
        "counterfactual_attribution"
    ]

    geo = p[
        "spatial_geometry"
    ]


    # Records exactly which numerical packet facts
    # were used in the rendered answer.
    used = {}


    # ========================================================
    # 1. FLOOD EXTENT
    # ========================================================

    if plan == "PLAN_EXTENT":

        temp_fraction = finite_number_or_none(
            seg[
                "temporary_flood"
            ][
                "fraction_of_coverage"
            ]
        )

        temp_area = finite_number_or_none(
            seg[
                "temporary_flood"
            ].get(
                "area_km2"
            )
        )

        perm_fraction = finite_number_or_none(
            seg[
                "permanent_water"
            ][
                "fraction_of_coverage"
            ]
        )


        used = {
            "temp_fraction": temp_fraction,
            "temp_area_km2": temp_area,
            "perm_fraction": perm_fraction,
        }


        text = (
            f"Temporary flooding covers "
            f"{pct(temp_fraction)} "
            f"of the sensor-covered area "
            f"({area(temp_area)}). "
            f"Permanent water covers "
            f"{pct(perm_fraction)}."
        )


    # ========================================================
    # 2. UNCERTAINTY
    # ========================================================

    elif plan == "PLAN_UNCERTAINTY":

        entropy = finite_number_or_none(
            unc.get(
                "mean_entropy"
            )
        )

        review10 = finite_number_or_none(
            unc.get(
                "review10_fraction"
            )
        )

        review20 = finite_number_or_none(
            unc.get(
                "review20_fraction"
            )
        )

        pclear = finite_number_or_none(
            sq.get(
                "mean_pclear"
            )
        )


        used = {
            "entropy": entropy,
            "review10": review10,
            "review20": review20,
            "pclear": pclear,
        }


        text = (
            f"Mean normalized predictive entropy is "
            f"{dec(entropy)}. "
            f"The top review fractions are "
            f"{pct(review10)} and "
            f"{pct(review20)}; "
            f"mean P(clear) is "
            f"{dec(pclear)}. "
            f"These are predictive-reliability indicators, "
            f"not Bayesian epistemic uncertainty."
        )


    # ========================================================
    # 3. GLOBAL SENSOR ATTRIBUTION
    # ========================================================

    elif plan == "PLAN_SENSOR_ATTRIBUTION":

        optical_share = finite_number_or_none(
            att.get(
                "optical_share"
            )
        )

        sar_share = finite_number_or_none(
            att.get(
                "sar_share"
            )
        )

        flip_no_optical = finite_number_or_none(
            att.get(
                "flip_rate_no_optical"
            )
        )

        flip_no_sar = finite_number_or_none(
            att.get(
                "flip_rate_no_sar"
            )
        )

        dominance = str(
            att.get(
                "dominance",
                "unavailable",
            )
        )


        used = {
            "optical_share": optical_share,
            "sar_share": sar_share,
            "flip_no_optical": flip_no_optical,
            "flip_no_sar": flip_no_sar,
        }


        text = (
            f"Counterfactual dependence is "
            f"{dominance}. "
            f"Optical share is "
            f"{pct(optical_share)} "
            f"and SAR share is "
            f"{pct(sar_share)}. "
            f"Removing optical flips "
            f"{pct(flip_no_optical)} "
            f"of predictions, while removing SAR flips "
            f"{pct(flip_no_sar)}. "
            f"These are model-dependence measures, "
            f"not physical causality."
        )


    # ========================================================
    # 4. TEMPORARY-FLOOD SENSOR ATTRIBUTION
    #
    # IMPORTANT:
    # These values can legitimately be unavailable.
    #
    # None -> deterministic abstention.
    # ========================================================

    elif plan == "PLAN_TEMP_ATTRIBUTION":

        raw_optical = att.get(
            "temporary_flood_js_optical"
        )

        raw_sar = att.get(
            "temporary_flood_js_sar"
        )


        temp_js_optical = finite_number_or_none(
            raw_optical
        )

        temp_js_sar = finite_number_or_none(
            raw_sar
        )


        # ----------------------------------------------------
        # Both values available
        # ----------------------------------------------------

        if (
            temp_js_optical is not None
            and temp_js_sar is not None
        ):

            used = {
                "temp_js_optical": temp_js_optical,
                "temp_js_sar": temp_js_sar,
            }


            text = (
                f"For temporary-flood predictions, "
                f"normalized JS dependence is "
                f"{dec(temp_js_optical)} "
                f"for optical removal and "
                f"{dec(temp_js_sar)} "
                f"for SAR removal. "
                f"These values describe model sensitivity, "
                f"not physical causality."
            )


        # ----------------------------------------------------
        # Optical only
        # ----------------------------------------------------

        elif (
            temp_js_optical is not None
            and temp_js_sar is None
        ):

            used = {
                "temp_js_optical": temp_js_optical,
                "temp_js_sar": None,
            }


            text = (
                f"Temporary-flood-specific attribution is "
                f"only partially available for this scene. "
                f"The normalized JS dependence for optical "
                f"removal is {dec(temp_js_optical)}; "
                f"the corresponding SAR-removal value is "
                f"unavailable. "
                f"FloodShift does not infer a missing value. "
                f"This is a model-dependence measure, "
                f"not physical causality."
            )


        # ----------------------------------------------------
        # SAR only
        # ----------------------------------------------------

        elif (
            temp_js_optical is None
            and temp_js_sar is not None
        ):

            used = {
                "temp_js_optical": None,
                "temp_js_sar": temp_js_sar,
            }


            text = (
                f"Temporary-flood-specific attribution is "
                f"only partially available for this scene. "
                f"The normalized JS dependence for SAR "
                f"removal is {dec(temp_js_sar)}; "
                f"the corresponding optical-removal value is "
                f"unavailable. "
                f"FloodShift does not infer a missing value. "
                f"This is a model-dependence measure, "
                f"not physical causality."
            )


        # ----------------------------------------------------
        # Neither value available
        # ----------------------------------------------------

        else:

            used = {
                "temp_js_optical": None,
                "temp_js_sar": None,
            }


            text = (
                f"Temporary-flood-specific counterfactual "
                f"attribution is unavailable for this scene. "
                f"FloodShift therefore abstains from reporting "
                f"temporary-flood SAR/optical JS values rather "
                f"than inferring unsupported numbers."
            )


    # ========================================================
    # 5. SPATIAL GEOMETRY
    # ========================================================

    elif plan == "PLAN_SPATIAL_GEOMETRY":

        components = safe_get(
            geo,
            "temporary_flood_connected_components_8conn",
            default=None,
        )

        largest_pixels = safe_get(
            geo,
            "largest_temporary_component_pixels",
            default=None,
        )

        largest_area = safe_get(
            geo,
            "largest_temporary_component_area_km2",
            default=None,
        )


        components = finite_number_or_none(
            components
        )

        largest_pixels = finite_number_or_none(
            largest_pixels
        )

        largest_area = finite_number_or_none(
            largest_area
        )


        used = {}


        bits = []


        if components is not None:

            used[
                "components"
            ] = components

            bits.append(
                f"The temporary-flood mask contains "
                f"{int(components)} raw 8-connected components."
            )

        else:

            bits.append(
                "The temporary-flood connected-component count "
                "is unavailable for this scene."
            )


        if largest_pixels is not None:

            used[
                "largest_component_pixels"
            ] = largest_pixels

            bits.append(
                f"The largest component contains "
                f"{int(largest_pixels)} pixels."
            )


        if largest_area is not None:

            used[
                "largest_component_area_km2"
            ] = largest_area

            bits.append(
                f"Its area is "
                f"{area(largest_area)}."
            )


        bits.append(
            "Connected components are pixel-mask structures, "
            "not separate flood events."
        )


        text = " ".join(
            bits
        )


    # ========================================================
    # 6. INTEGRATED SUMMARY
    # ========================================================

    elif plan == "PLAN_INTEGRATED_SUMMARY":

        temp_fraction = finite_number_or_none(
            seg[
                "temporary_flood"
            ].get(
                "fraction_of_coverage"
            )
        )

        temp_area = finite_number_or_none(
            seg[
                "temporary_flood"
            ].get(
                "area_km2"
            )
        )

        entropy = finite_number_or_none(
            unc.get(
                "mean_entropy"
            )
        )

        pclear = finite_number_or_none(
            sq.get(
                "mean_pclear"
            )
        )

        optical_share = finite_number_or_none(
            att.get(
                "optical_share"
            )
        )

        sar_share = finite_number_or_none(
            att.get(
                "sar_share"
            )
        )


        used = {
            "temp_fraction": temp_fraction,
            "temp_area": temp_area,
            "entropy": entropy,
            "pclear": pclear,
            "optical_share": optical_share,
            "sar_share": sar_share,
        }


        text = (
            f"FloodShift predicts temporary flooding over "
            f"{pct(temp_fraction)} "
            f"of covered area "
            f"({area(temp_area)}). "
            f"Mean predictive entropy is "
            f"{dec(entropy)}, "
            f"mean P(clear) is "
            f"{dec(pclear)}, "
            f"and counterfactual dependence is "
            f"{pct(optical_share)} optical versus "
            f"{pct(sar_share)} SAR."
        )


    # ========================================================
    # 7. FIXED REVIEW POLICY
    # ========================================================

    elif plan == "PLAN_REVIEW":

        review = val_review.get(
            scene_id,
            train_review.get(
                scene_id,
                0,
            ),
        )


        review10 = finite_number_or_none(
            unc.get(
                "review10_fraction"
            )
        )

        high_fragility = finite_number_or_none(
            att.get(
                "high_fragility_tile_fraction"
            )
        )


        used = {
            "review10": review10,
            "high_fragility": high_fragility,
        }


        if review:

            prefix = (
                "Yes. The fixed FloodShift policy "
                "flags this scene for review."
            )

        else:

            prefix = (
                "No. The fixed FloodShift policy "
                "does not flag this scene for review."
            )


        text = (
            prefix
            + f" Review10 fraction is "
            + f"{pct(review10)} "
            + f"and high-fragility tile fraction is "
            + f"{pct(high_fragility)}."
        )


    # ========================================================
    # 8. PHYSICAL-CAUSE REFUSAL
    # ========================================================

    elif plan == "REFUSE_PHYSICAL_CAUSE":

        used = {}

        text = (
            "FloodShift does not establish the physical cause "
            "of a flood. Counterfactual sensor attribution "
            "measures model dependence, not physical causality."
        )


    # ========================================================
    # 9. HUMAN-IMPACT REFUSAL
    # ========================================================

    elif plan == "REFUSE_HUMAN_IMPACT":

        used = {}

        text = (
            "FloodShift does not contain verified population, "
            "casualty, property-damage, or human-impact data, "
            "so it cannot determine those impacts from this evidence."
        )


    # ========================================================
    # 10. FUTURE-FORECAST REFUSAL
    # ========================================================

    elif plan == "REFUSE_FUTURE_FORECAST":

        used = {}

        text = (
            "FloodShift's current evidence packet does not provide "
            "a future flood-propagation forecast. "
            "A forecast would require external temporal or "
            "hydrologic evidence."
        )


    # ========================================================
    # 11. VISUAL-ONLY DESCRIPTORS
    # ========================================================

    elif plan.startswith(
        "VISUAL_"
    ):

        # These are discrete VLM descriptors.
        # They intentionally contain no numerical scientific claims.
        used = {}

        text = plan


    # ========================================================
    # UNKNOWN PLAN
    # ========================================================

    else:

        raise KeyError(
            f"Unknown plan: {plan}"
        )


    # ========================================================
    # RETURN VERIFIED RESPONSE
    # ========================================================

    return {
        "text": text,
        "used_facts": used,
    }


# ============================================================
# BASIC SANITY CHECK
# ============================================================

assert callable(
    render_plan
)

print("=" * 112)
print("P20-C19 — DETERMINISTIC VERIFIED RENDERER")
print("=" * 112)

print(
    "Missing temporary-attribution handling : ENABLED"
)

print(
    "Unsupported-number inference           : DISABLED"
)

print(
    "Deterministic abstention               : ENABLED"
)

print(
    "Geo test                               : LOCKED_NOT_EVALUATED"
)

print()

print(
    "P20-C19 COMPLETE ✅"
)

P20-C19 — DETERMINISTIC VERIFIED RENDERER
Missing temporary-attribution handling : ENABLED
Unsupported-number inference           : DISABLED
Deterministic abstention               : ENABLED
Geo test                               : LOCKED_NOT_EVALUATED

P20-C19 COMPLETE ✅


In [37]:
# ============================================================
# P20-C20 — FINAL SYSTEM EVALUATION
# Corrected + robust version
# ============================================================

# ------------------------------------------------------------
# Use final FULL planner predictions
# ------------------------------------------------------------

full_df = planner_full[
    "df"
].copy()


# ------------------------------------------------------------
# Render every predicted plan deterministically
# ------------------------------------------------------------

render_rows = []

for _, row in full_df.iterrows():

    rendered = render_plan(
        row["scene_id"],
        row["pred_plan"],
    )

    render_rows.append({

        **row.to_dict(),

        "rendered_text": rendered[
            "text"
        ],

        "used_facts": rendered[
            "used_facts"
        ],

        "plan_correct": (
            int(row["pred_id"])
            == int(row["target_id"])
        ),
    })


render_df = pd.DataFrame(
    render_rows
)


assert len(render_df) == len(full_df), (
    f"Render row mismatch: "
    f"{len(render_df)} vs {len(full_df)}"
)


# ============================================================
# SPLIT GROUNDED VS VISUAL-ONLY
# ============================================================

grounded_df = render_df[
    ~render_df[
        "is_visual_only"
    ].astype(bool)
].copy()


visual_df = render_df[
    render_df[
        "is_visual_only"
    ].astype(bool)
].copy()


assert len(grounded_df) > 0
assert len(visual_df) > 0


# ============================================================
# PLAN ACCURACY
# ============================================================

grounded_plan_accuracy = float(
    grounded_df[
        "plan_correct"
    ].mean()
)


visual_plan_accuracy = float(
    visual_df[
        "plan_correct"
    ].mean()
)


# ============================================================
# REFUSAL METRICS
#
# IMPORTANT:
# Here refusal is identified from the PLAN CLASS itself,
# not from phrases such as "does not".
#
# This fixes the evaluator problem from 09A/09B.
# ============================================================

refusal_target = (
    grounded_df[
        "target_plan"
    ]
    .astype(str)
    .str.startswith(
        "REFUSE_"
    )
)


refusal_pred = (
    grounded_df[
        "pred_plan"
    ]
    .astype(str)
    .str.startswith(
        "REFUSE_"
    )
)


# ------------------------------------------------------------
# Refusal recall
# ------------------------------------------------------------

if refusal_target.any():

    refusal_recall = float(
        refusal_pred[
            refusal_target
        ].mean()
    )

else:

    refusal_recall = float("nan")


# ------------------------------------------------------------
# False-refusal rate on answer tasks
# ------------------------------------------------------------

answer_mask = (
    ~refusal_target
)

if answer_mask.any():

    false_refusal_rate = float(
        refusal_pred[
            answer_mask
        ].mean()
    )

else:

    false_refusal_rate = float("nan")


# ============================================================
# DETERMINISTIC RENDERER FACT AUDIT
#
# Missing facts represented as None are VALID abstentions.
#
# Only a non-finite value that was actually claimed counts
# as a renderer fact violation.
# ============================================================

renderer_fact_violations = 0

renderer_checked_numeric_facts = 0

renderer_missing_facts = 0


for _, row in render_df.iterrows():

    facts = row[
        "used_facts"
    ]

    # Defensive handling
    if facts is None:
        continue

    if not isinstance(
        facts,
        dict,
    ):
        raise TypeError(
            f"used_facts must be dict, "
            f"got {type(facts)}"
        )

    for key, value in facts.items():

        # Missing evidence is allowed.
        # Renderer explicitly abstains.
        if value is None:

            renderer_missing_facts += 1

            continue


        renderer_checked_numeric_facts += 1


        try:

            number = float(
                value
            )

        except (
            TypeError,
            ValueError,
        ):

            renderer_fact_violations += 1

            print(
                "⚠️ Non-numeric rendered fact:",
                row["scene_id"],
                row["pred_plan"],
                key,
                value,
            )

            continue


        if not np.isfinite(
            number
        ):

            renderer_fact_violations += 1

            print(
                "⚠️ Non-finite rendered fact:",
                row["scene_id"],
                row["pred_plan"],
                key,
                value,
            )


# ============================================================
# PER-TASK PLAN ACCURACY
# ============================================================

per_task_plan = (
    render_df
    .groupby(
        "task",
        as_index=False,
    )
    .agg(
        examples=(
            "id",
            "count",
        ),

        plan_accuracy=(
            "plan_correct",
            "mean",
        ),
    )
)


# ============================================================
# EXTRA DIAGNOSTICS
# ============================================================

overall_plan_accuracy = float(
    render_df[
        "plan_correct"
    ].mean()
)


grounded_correct = int(
    grounded_df[
        "plan_correct"
    ].sum()
)


visual_correct = int(
    visual_df[
        "plan_correct"
    ].sum()
)


refusal_examples = int(
    refusal_target.sum()
)


answer_examples = int(
    answer_mask.sum()
)


# ============================================================
# REPORT
# ============================================================

print("=" * 112)
print("P20-C20 — FINAL SYSTEM METRICS")
print("=" * 112)


print()
print("GAUSSIAN UNCERTAINTY")
print("-" * 60)

print(
    "Entropy correlation        :",
    gaussian_entropy_r,
)

print(
    "Entropy MAE                :",
    gaussian_entropy_mae,
)


print()
print("VISUAL BRIDGE")
print("-" * 60)

print(
    "Full core balanced acc     :",
    full_core,
)

print(
    "Zero core balanced acc     :",
    zero_core,
)

print(
    "Shuffled core balanced acc :",
    shuffle_core,
)

print(
    "Full - zero gap            :",
    full_core
    - zero_core,
)

print(
    "Full - shuffled gap        :",
    full_core
    - shuffle_core,
)


print()
print("PLANNER")
print("-" * 60)

print(
    "Overall plan accuracy      :",
    overall_plan_accuracy,
)

print(
    "Grounded plan accuracy     :",
    grounded_plan_accuracy,
)

print(
    "Grounded correct           :",
    f"{grounded_correct}/{len(grounded_df)}",
)

print(
    "Visual plan accuracy       :",
    visual_plan_accuracy,
)

print(
    "Visual correct             :",
    f"{visual_correct}/{len(visual_df)}",
)

print(
    "Visual full-zero gap       :",
    planner_full[
        "visual_accuracy"
    ]
    - planner_zero[
        "visual_accuracy"
    ],
)

print(
    "Visual full-shuffle gap    :",
    planner_full[
        "visual_accuracy"
    ]
    - planner_shuffle[
        "visual_accuracy"
    ],
)


print()
print("REFUSAL / SAFETY")
print("-" * 60)

print(
    "Refusal examples           :",
    refusal_examples,
)

print(
    "Answer examples            :",
    answer_examples,
)

print(
    "Refusal recall             :",
    refusal_recall,
)

print(
    "False refusal rate         :",
    false_refusal_rate,
)


print()
print("DETERMINISTIC RENDERER")
print("-" * 60)

print(
    "Checked numeric facts      :",
    renderer_checked_numeric_facts,
)

print(
    "Unavailable facts abstained:",
    renderer_missing_facts,
)

print(
    "Renderer fact violations   :",
    renderer_fact_violations,
)


print()
print("PROTOCOL")
print("-" * 60)

print(
    "Geo test                   :",
    "LOCKED_NOT_EVALUATED",
)


print()
print("PER-TASK PLAN ACCURACY")
print("-" * 60)

display(
    per_task_plan
)


# ============================================================
# FINAL SANITY ASSERTIONS
# ============================================================

assert np.isfinite(
    grounded_plan_accuracy
)

assert np.isfinite(
    visual_plan_accuracy
)

assert renderer_fact_violations >= 0


print()
print("P20-C20 COMPLETE ✅")

P20-C20 — FINAL SYSTEM METRICS

GAUSSIAN UNCERTAINTY
------------------------------------------------------------
Entropy correlation        : 0.8414468577487988
Entropy MAE                : 0.0197257731548127

VISUAL BRIDGE
------------------------------------------------------------
Full core balanced acc     : 0.7387254901960785
Zero core balanced acc     : 0.3333333333333333
Shuffled core balanced acc : 0.2936507936507936
Full - zero gap            : 0.40539215686274516
Full - shuffled gap        : 0.44507469654528486

PLANNER
------------------------------------------------------------
Overall plan accuracy      : 0.9159663865546218
Grounded plan accuracy     : 1.0
Grounded correct           : 340/340
Visual plan accuracy       : 0.7058823529411765
Visual correct             : 96/136
Visual full-zero gap       : 0.19852941176470595
Visual full-shuffle gap    : 0.23529411764705888

REFUSAL / SAFETY
------------------------------------------------------------
Refusal examples       

,task,examples,plan_accuracy
0,cause_refusal,34,1.000000
1,extent,34,1.000000
2,forecast_refusal,34,1.000000
3,impact_refusal,34,1.000000
4,integrated_summary,34,1.000000
5,review_policy,34,1.000000
6,sensor_attribution,34,1.000000
7,spatial_geometry,34,1.000000
8,temporary_sensor_attribution,34,1.000000
9,uncertainty,34,1.000000



P20-C20 COMPLETE ✅


In [38]:
# ============================================================
# P20-C22 — QUALITATIVE EXAMPLES
# ============================================================
for task in [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "temporary_sensor_attribution",
    "spatial_geometry",
    "integrated_summary",
    "review_policy",
    "cause_refusal",
    "impact_refusal",
    "forecast_refusal",
    "visual_extent",
    "visual_uncertainty",
]:
    rows = render_df[
        render_df[
            "task"
        ]
        == task
    ]

    if len(
        rows
    ) == 0:
        continue

    row = rows.iloc[
        0
    ]

    print("=" * 112)
    print("TASK       :", task)
    print("SCENE      :", row["scene_id"])
    print("TARGET PLAN:", row["target_plan"])
    print("PRED PLAN  :", row["pred_plan"])
    print("CORRECT    :", row["plan_correct"])
    print()
    print("FINAL RENDERED RESPONSE:")
    print(row["rendered_text"])
    print()

print("P20-C22 COMPLETE ✅")


TASK       : extent
SCENE      : Somalia_12849
TARGET PLAN: PLAN_EXTENT
PRED PLAN  : PLAN_EXTENT
CORRECT    : True

FINAL RENDERED RESPONSE:
Temporary flooding covers 20.68% of the sensor-covered area (5.38 km²). Permanent water covers 0.07%.

TASK       : uncertainty
SCENE      : Somalia_12849
TARGET PLAN: PLAN_UNCERTAINTY
PRED PLAN  : PLAN_UNCERTAINTY
CORRECT    : True

FINAL RENDERED RESPONSE:
Mean normalized predictive entropy is 0.186. The top review fractions are 38.72% and 72.95%; mean P(clear) is 0.498. These are predictive-reliability indicators, not Bayesian epistemic uncertainty.

TASK       : sensor_attribution
SCENE      : Somalia_12849
TARGET PLAN: PLAN_SENSOR_ATTRIBUTION
PRED PLAN  : PLAN_SENSOR_ATTRIBUTION
CORRECT    : True

FINAL RENDERED RESPONSE:
Counterfactual dependence is Optical-dominant. Optical share is 85.64% and SAR share is 14.36%. Removing optical flips 20.75% of predictions, while removing SAR flips 5.99%. These are model-dependence measures, not physical 

In [41]:
# ============================================================
# P20-C23 — EXPORT + ARCHIVE
# ============================================================
# Save compact trainable modules only.
LORA_DIR = (
    CHECKPOINT_ROOT
    / "flan_t5_encoder_lora"
)

text_encoder.save_pretrained(
    LORA_DIR
)

TOKENIZER_DIR = (
    CHECKPOINT_ROOT
    / "tokenizer"
)

tokenizer.save_pretrained(
    TOKENIZER_DIR
)

PLAN_HEAD_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_10_PlanHead.pt"
)

torch.save(
    {
        "state_dict": {
            k: v.detach().cpu()
            for k, v in plan_head.state_dict().items()
        },
        "plan_vocab": PLAN_VOCAB,
        "text_dim": TEXT_DIM,
    },
    PLAN_HEAD_PATH,
)

FINAL_VISUAL_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_10_FinalVisualBridge.pt"
)

torch.save(
    {
        "state_dict": {
            k: v.detach().cpu()
            for k, v in visual_bridge.state_dict().items()
        },
        "highres_vision": HIGHRES_VISION_NAME,
        "highres_dim": HIGHRES_DIM,
        "text_dim": TEXT_DIM,
        "latent_dim": LATENT_DIM,
        "extent_bounds": EXTENT_BOUNDS,
        "uncertainty_bounds": UNCERTAINTY_BOUNDS,
        "best_visual_epoch": best_visual_epoch,
    },
    FINAL_VISUAL_PATH,
)

# Metrics.
full_visual_task.to_csv(
    METRIC_ROOT
    / "FloodShift_10_visual_full_per_task.csv",
    index=False,
)

summary.to_csv(
    METRIC_ROOT
    / "FloodShift_10_visual_ablation.csv",
    index=False,
)

planner_ablation.to_csv(
    METRIC_ROOT
    / "FloodShift_10_planner_ablation.csv",
    index=False,
)

per_task_plan.to_csv(
    METRIC_ROOT
    / "FloodShift_10_plan_per_task.csv",
    index=False,
)

gate_df.to_csv(
    METRIC_ROOT
    / "FloodShift_10_acceptance_gates.csv",
    index=False,
)

render_export = render_df.drop(
    columns=[
        "used_facts",
    ]
).copy()

render_export.to_csv(
    METRIC_ROOT
    / "FloodShift_10_validation_rendered.csv",
    index=False,
)

with open(
    METRIC_ROOT
    / "FloodShift_10_visual_history.json",
    "w",
) as f:
    json.dump(
        visual_history,
        f,
        indent=2,
    )

with open(
    METRIC_ROOT
    / "FloodShift_10_planner_history.json",
    "w",
) as f:
    json.dump(
        planner_history,
        f,
        indent=2,
    )

PROTOCOL_PATH = (
    OUTPUT_ROOT
    / "FloodShift_10_protocol_record.json"
)

protocol10 = {
    "experiment": "FloodShift-10-Final-Reliable-Grounded-Variational-VLP",
    "source_08b_sha256": EXPECTED_08B_SHA,
    "source_09b_sha256": EXPECTED_09B_SHA,
    "architecture": {
        "high_resolution_visual_encoder": HIGHRES_VISION_NAME,
        "high_resolution_visual_encoder_trainable": False,
        "gaussian_uncertainty_encoder": "Frozen successful FloodShift-09B GaussianSceneEncoder",
        "gaussian_trainable": False,
        "instruction_aware_visual_resampler": True,
        "query_tokens": N_QUERIES,
        "text_encoder": TEXT_ENCODER_NAME,
        "text_adaptation": "LoRA q/v",
        "output": "finite response plan + deterministic fact renderer",
    },
    "train_only_bins": {
        "extent_tertiles": list(
            EXTENT_BOUNDS
        ),
        "uncertainty_tertiles": list(
            UNCERTAINTY_BOUNDS
        ),
    },
    "metrics": {
        "gaussian_entropy_correlation": gaussian_entropy_r,
        "gaussian_entropy_mae": gaussian_entropy_mae,
        "visual_core_balanced_accuracy": full_core,
        "visual_full_minus_zero": full_core - zero_core,
        "visual_full_minus_shuffle": full_core - shuffle_core,
        "grounded_plan_accuracy": grounded_plan_accuracy,
        "visual_plan_accuracy": visual_plan_accuracy,
        "planner_visual_full_minus_zero": (
            planner_full["visual_accuracy"]
            - planner_zero["visual_accuracy"]
        ),
        "planner_visual_full_minus_shuffle": (
            planner_full["visual_accuracy"]
            - planner_shuffle["visual_accuracy"]
        ),
        "refusal_recall": refusal_recall,
        "false_refusal_rate": false_refusal_rate,
        "renderer_fact_violations": renderer_fact_violations,
    },
    "acceptance_gates": {
        k: bool(v)
        for k, v in FINAL_GATES.items()
    },
    "status": (
        "ACCEPTED"
        if FINAL_ACCEPTED
        else "NOT_ACCEPTED"
    ),
    "frozen_geographic_test": "LOCKED_NOT_EVALUATED",
}

with open(
    PROTOCOL_PATH,
    "w",
) as f:
    json.dump(
        protocol10,
        f,
        indent=2,
    )

def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()

MANIFEST_PATH = (
    OUTPUT_ROOT
    / "FloodShift_10_file_manifest.csv"
)

manifest_rows = []

for root in [
    CHECKPOINT_ROOT,
    METRIC_ROOT,
]:
    for p in sorted(
        root.rglob("*")
    ):
        if p.is_file():
            manifest_rows.append({
                "relative_path": str(
                    p.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "bytes": int(
                    p.stat().st_size
                ),
                "sha256": file_sha256(
                    p
                ),
            })

manifest_rows.append({
    "relative_path": str(
        PROTOCOL_PATH.relative_to(
            OUTPUT_ROOT
        )
    ),
    "bytes": int(
        PROTOCOL_PATH.stat().st_size
    ),
    "sha256": file_sha256(
        PROTOCOL_PATH
    ),
})

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
)

ARCHIVE_PATH = (
    OUTPUT_ROOT
    / "FloodShift_10_ARCHIVE.zip"
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for root in [
        CHECKPOINT_ROOT,
        METRIC_ROOT,
    ]:
        for p in sorted(
            root.rglob("*")
        ):
            if p.is_file():
                zf.write(
                    p,
                    arcname=p.relative_to(
                        OUTPUT_ROOT
                    ),
                )

    zf.write(
        PROTOCOL_PATH,
        arcname=PROTOCOL_PATH.relative_to(
            OUTPUT_ROOT
        ),
    )

    zf.write(
        MANIFEST_PATH,
        arcname=MANIFEST_PATH.relative_to(
            OUTPUT_ROOT
        ),
    )

ARCHIVE_SHA = file_sha256(
    ARCHIVE_PATH
)

print("=" * 112)
print("P20-C23 — FINAL ARCHIVE")
print("=" * 112)
print("Status          :", protocol10["status"])
print("Gaussian r      :", gaussian_entropy_r)
print("Visual core     :", full_core)
print("Visual zero gap :", full_core - zero_core)
print("Visual shuf gap :", full_core - shuffle_core)
print("Grounded plan   :", grounded_plan_accuracy)
print("Visual plan     :", visual_plan_accuracy)
print("Refusal recall  :", refusal_recall)
print("False refusal   :", false_refusal_rate)
print("Fact violations :", renderer_fact_violations)
print("Geo test        : LOCKED_NOT_EVALUATED")
print("Archive         :", ARCHIVE_PATH)
print("Archive SHA256  :", ARCHIVE_SHA)
print()
print("Upload the executed notebook for final acceptance analysis.")
print("P20-C23 COMPLETE ✅")


P20-C23 — FINAL ARCHIVE
Status          : ACCEPTED
Gaussian r      : 0.8414468577487988
Visual core     : 0.7387254901960785
Visual zero gap : 0.40539215686274516
Visual shuf gap : 0.44507469654528486
Grounded plan   : 1.0
Visual plan     : 0.7058823529411765
Refusal recall  : 1.0
False refusal   : 0.0
Fact violations : 0
Geo test        : LOCKED_NOT_EVALUATED
Archive         : /kaggle/working/FloodShift_10_Final_VLP/FloodShift_10_ARCHIVE.zip
Archive SHA256  : bbe438ff28ab8f3b8785907b9ba92ca7f45e36f07971e9faa26a4f96c3789123

Upload the executed notebook for final acceptance analysis.
P20-C23 COMPLETE ✅


In [40]:
# ============================================================
# P20-C21 — FINAL ACCEPTANCE GATES
# Self-contained recovery version
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Sanity: C20 must have completed
# ------------------------------------------------------------

required_vars = [
    "gaussian_entropy_r",
    "full_core",
    "zero_core",
    "shuffle_core",
    "grounded_plan_accuracy",
    "visual_plan_accuracy",
    "planner_full",
    "planner_zero",
    "planner_shuffle",
    "refusal_recall",
    "false_refusal_rate",
    "renderer_fact_violations",
]

missing = [
    x
    for x in required_vars
    if x not in globals()
]

assert not missing, (
    "P20-C20 has not completed successfully.\n"
    f"Missing variables: {missing}\n"
    "Run corrected P20-C20 first."
)


# ============================================================
# DERIVED VISUAL GAPS
# ============================================================

visual_bridge_full_zero_gap = (
    float(full_core)
    - float(zero_core)
)

visual_bridge_full_shuffle_gap = (
    float(full_core)
    - float(shuffle_core)
)


planner_visual_full_zero_gap = (
    float(
        planner_full[
            "visual_accuracy"
        ]
    )
    - float(
        planner_zero[
            "visual_accuracy"
        ]
    )
)


planner_visual_full_shuffle_gap = (
    float(
        planner_full[
            "visual_accuracy"
        ]
    )
    - float(
        planner_shuffle[
            "visual_accuracy"
        ]
    )
)


# ============================================================
# PREDECLARED ACCEPTANCE GATES
# ============================================================

FINAL_GATES = {

    # --------------------------------------------------------
    # Gaussian uncertainty
    # --------------------------------------------------------

    "gaussian_entropy_r_ge_0.30": (
        float(
            gaussian_entropy_r
        )
        >= 0.30
    ),


    # --------------------------------------------------------
    # Core visual grounding
    # --------------------------------------------------------

    "core_visual_balanced_accuracy_ge_0.70": (
        float(
            full_core
        )
        >= 0.70
    ),

    "core_visual_full_minus_zero_ge_0.15": (
        visual_bridge_full_zero_gap
        >= 0.15
    ),

    "core_visual_full_minus_shuffle_ge_0.15": (
        visual_bridge_full_shuffle_gap
        >= 0.15
    ),


    # --------------------------------------------------------
    # Planner
    # --------------------------------------------------------

    "grounded_plan_accuracy_ge_0.95": (
        float(
            grounded_plan_accuracy
        )
        >= 0.95
    ),

    "visual_plan_accuracy_ge_0.70": (
        float(
            visual_plan_accuracy
        )
        >= 0.70
    ),

    "planner_visual_full_minus_zero_ge_0.10": (
        planner_visual_full_zero_gap
        >= 0.10
    ),

    "planner_visual_full_minus_shuffle_ge_0.10": (
        planner_visual_full_shuffle_gap
        >= 0.10
    ),


    # --------------------------------------------------------
    # Refusal / safety
    # --------------------------------------------------------

    "refusal_recall_ge_0.95": (
        np.isfinite(
            refusal_recall
        )
        and float(
            refusal_recall
        )
        >= 0.95
    ),

    "false_refusal_le_0.05": (
        np.isfinite(
            false_refusal_rate
        )
        and float(
            false_refusal_rate
        )
        <= 0.05
    ),


    # --------------------------------------------------------
    # Deterministic fact integrity
    # --------------------------------------------------------

    "renderer_fact_violations_eq_0": (
        int(
            renderer_fact_violations
        )
        == 0
    ),
}


# ============================================================
# GATE TABLE
# ============================================================

gate_df = pd.DataFrame(
    [
        {
            "gate": gate,
            "pass": bool(
                passed
            ),
        }
        for gate, passed
        in FINAL_GATES.items()
    ]
)


FINAL_ACCEPTED = bool(
    all(
        FINAL_GATES.values()
    )
)


# ============================================================
# REPORT
# ============================================================

print("=" * 112)
print("P20-C21 — FINAL ACCEPTANCE GATES")
print("=" * 112)

display(
    gate_df
)


print()
print("KEY VALUES")
print("-" * 60)

print(
    "Gaussian entropy r             :",
    float(
        gaussian_entropy_r
    ),
)

print(
    "Core visual balanced accuracy :",
    float(
        full_core
    ),
)

print(
    "Core full-zero gap            :",
    visual_bridge_full_zero_gap,
)

print(
    "Core full-shuffle gap         :",
    visual_bridge_full_shuffle_gap,
)

print(
    "Grounded plan accuracy        :",
    float(
        grounded_plan_accuracy
    ),
)

print(
    "Visual plan accuracy          :",
    float(
        visual_plan_accuracy
    ),
)

print(
    "Planner visual full-zero gap  :",
    planner_visual_full_zero_gap,
)

print(
    "Planner visual shuffle gap    :",
    planner_visual_full_shuffle_gap,
)

print(
    "Refusal recall                :",
    float(
        refusal_recall
    ),
)

print(
    "False refusal rate            :",
    float(
        false_refusal_rate
    ),
)

print(
    "Renderer fact violations      :",
    int(
        renderer_fact_violations
    ),
)


print()
print("=" * 112)

print(
    "FLOODSHIFT-10 STATUS:",
    (
        "ACCEPTED ✅"
        if FINAL_ACCEPTED
        else "NOT_ACCEPTED ❌"
    ),
)

print(
    "Frozen geographic test:",
    "LOCKED_NOT_EVALUATED",
)

print("=" * 112)

print()
print("P20-C21 COMPLETE ✅")

P20-C21 — FINAL ACCEPTANCE GATES


,gate,pass
0,gaussian_entropy_r_ge_0.30,True
1,core_visual_balanced_accuracy_ge_0.70,True
2,core_visual_full_minus_zero_ge_0.15,True
3,core_visual_full_minus_shuffle_ge_0.15,True
4,grounded_plan_accuracy_ge_0.95,True
5,visual_plan_accuracy_ge_0.70,True
6,planner_visual_full_minus_zero_ge_0.10,True
7,planner_visual_full_minus_shuffle_ge_0.10,True
8,refusal_recall_ge_0.95,True
9,false_refusal_le_0.05,True



KEY VALUES
------------------------------------------------------------
Gaussian entropy r             : 0.8414468577487988
Core visual balanced accuracy : 0.7387254901960785
Core full-zero gap            : 0.40539215686274516
Core full-shuffle gap         : 0.44507469654528486
Grounded plan accuracy        : 1.0
Visual plan accuracy          : 0.7058823529411765
Planner visual full-zero gap  : 0.19852941176470595
Planner visual shuffle gap    : 0.23529411764705888
Refusal recall                : 1.0
False refusal rate            : 0.0
Renderer fact violations      : 0

FLOODSHIFT-10 STATUS: ACCEPTED ✅
Frozen geographic test: LOCKED_NOT_EVALUATED

P20-C21 COMPLETE ✅


## After this notebook

If all P20-C21 gates pass:

1. **Freeze this VLP.**
2. Build the verifier/API layer around the deterministic renderer.
3. Build FloodShift Live:
   - upload compatible scene,
   - segmentation,
   - uncertainty,
   - sensor attribution,
   - GPS exposure,
   - external live weather/alert context,
   - live LLM API for phrasing verified facts.
4. Freeze the entire system.
5. Only then unlock the 58-scene geographic test once.

### Important deployment interpretation

The live LLM API is allowed to explain verified structured facts, but:
- the segmentation model owns flood detection;
- the GeoEngine owns spatial/numerical facts;
- the GPS engine owns location intersection/distance;
- external APIs own real-world weather/alert context;
- the LLM owns wording, not truth.
